# PROMISE12 Whole-Gland Benchmark Notebook

This Google Colab notebook is a full end-to-end benchmark for evaluating **four fixed prostate MRI segmentation models** on **PROMISE12 whole-gland test data** with ground-truth masks available at runtime.

## Deliverables

- automatic test-data download inside Colab
- automatic model / repo / weight acquisition inside Colab
- robust case discovery and MRI-to-ground-truth pairing
- unified whole-gland binary-mask standardization in GT space
- per-case metrics with exactly:
  - Dice
  - Tversky (`alpha=0.3`, `beta=0.7`)
  - Non-intersecting prediction percentage
- a clean summary comparison table
- a report-friendly qualitative figure with layout `[MRI] [GT] [Prediction]`
- automatic PDF report export

## Fixed Models

1. **Dzaridis**  
   <https://github.com/dzaridis/MRI-Prostate-Gland-and-Zone-Segmentor>
2. **DeepInfer prostate segmenter**  
   <https://www.deepinfer.org/models/prostate-segmenter/>
3. **BAMF AIMI prostate MRI**  
   <https://github.com/bamf-health/aimi-prostate-mr>
4. **MONAI `prostate_mri_anatomy`**  
   <https://huggingface.co/MONAI/prostate_mri_anatomy/tree/0.3.5>

## Ground-Truth Configuration Note

The test-data URL is prefilled below. The ground-truth source can be provided in either of two ways:

- embedded in the extracted dataset itself, for example `Case00.mhd` beside `Case00_segmentation.mhd`
- explicitly through `GT_DATA_URL` or `GT_LOCAL_PATH`

This is important for PROMISE12-style MetaImage layouts because `.mhd` header files often travel with companion `.raw` payload files, and the segmentation masks may already live beside the MRI volumes in the same extracted folder.


## Notebook Structure

The notebook is organized as both a benchmark report and an engineering pipeline.

1. Runtime diagnostics and dependency installation
2. Runtime configuration
3. Dataset download, extraction, and inspection
4. Case pairing
5. Common utilities and metric definitions
6. Four model adapters with a unified interface
7. End-to-end evaluation loop
8. Per-case and summary result tables
9. Best-case selection for qualitative visualization
10. PDF report export

The implementation prioritizes deterministic behavior where practical, explicit geometry checks before scoring, and clear failure messages instead of silent fallbacks.


In [1]:
from pathlib import Path
import os
import platform
import subprocess
import sys

print("Python executable :", sys.executable)
print("Python version    :", sys.version.replace("\n", " "))
print("Platform          :", platform.platform())
print("Working directory :", Path.cwd())

try:
    import torch

    print("Torch version     :", torch.__version__)
    print("CUDA available    :", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device count :", torch.cuda.device_count())
        print("Current device    :", torch.cuda.current_device())
        print("Device name       :", torch.cuda.get_device_name(torch.cuda.current_device()))
except Exception as exc:
    print("Torch diagnostic could not be completed:", exc)

try:
    subprocess.run(["nvidia-smi"], check=False)
except Exception as exc:
    print("nvidia-smi not available:", exc)


Python executable : /usr/bin/python3
Python version    : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform          : Linux-6.6.113+-x86_64-with-glibc2.35
Working directory : /content
Torch version     : 2.10.0+cu128
CUDA available    : True
CUDA device count : 1
Current device    : 0
Device name       : NVIDIA A100-SXM4-40GB


## Environment Setup

This benchmark needs a broad Colab toolchain:

- Python scientific stack
- medical-imaging IO for NIfTI and NRRD
- `nnUNet` v1 for BAMF
- `nnUNetv2` plus Dzaridis dependencies
- MONAI bundle tools
- `udocker` for a Docker-daemon-free DeepInfer path
- ReportLab for PDF export

The DeepInfer adapter is the least notebook-native path because the official model is Docker-first. The notebook therefore attempts a real Colab-compatible container path via `udocker --allow-root` and preserves the rest of the benchmark if that setup fails.


In [2]:
import json
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path


INSTALL_SENTINEL = Path("/content/promise12_install_manifest.json")
ENVIRONMENT_SPEC = {
    "apt": [
        "git",
        "git-lfs",
        "pigz",
        "p7zip-full",
        "unzip",
        "libgl1",
        "libglib2.0-0",
    ],
    "pip": [
        "numpy==1.26.2",
        "scipy==1.13.1",
        "pandas==2.2.2",
        "matplotlib==3.8.4",
        "scikit-learn==1.5.2",
        "tqdm==4.66.4",
        "requests==2.32.3",
        "reportlab==4.2.2",
        "tabulate==0.9.0",
        "SimpleITK==2.3.1",
        "nibabel==5.2.1",
        "pynrrd==1.0.0",
        "pydicom==2.4.4",
        "pybase64==1.3.2",
        "httplib2==0.22.0",
        "huggingface_hub==0.23.4",
        "monai==1.4.0",
        "pytorch-ignite==0.4.11",
        "fire==0.6.0",
        "MedProIO==0.1.1",
        "batchgenerators==0.25",
        "scikit-image==0.24.0",
        "tifffile==2024.8.30",
        "medpy==0.5.2",
        "dicom2nifti==2.4.11",
        "PyYAML==6.0.2",
        "nnunetv2==2.2.1",
        "udocker>=1.3.17",
    ],
    "nnunet_v1": {
        "repo": "https://github.com/MIC-DKFZ/nnUNet.git",
        "path": "/content/nnUNet_v1",
        "branch": "v1.7.1",
    },
}
RESTART_SENSITIVE_MODULES = [
    "numpy",
    "scipy",
    "pandas",
    "matplotlib",
    "SimpleITK",
    "nibabel",
    "nrrd",
    "pydicom",
    "monai",
    "ignite",
    "nnunet",
    "nnunetv2",
]
loaded_before_install = [name for name in RESTART_SENSITIVE_MODULES if name in sys.modules]


def run_install(cmd, check=True, cwd=None, env=None):
    printable = cmd if isinstance(cmd, str) else " ".join(shlex.quote(str(x)) for x in cmd)
    print(f"\n$ {printable}")
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        check=False,
    )
    print(f"[exit code] {result.returncode}")
    if check and result.returncode != 0:
        raise RuntimeError(
            "Install step failed.\n"
            f"Command: {printable}\n"
            f"Exit code: {result.returncode}\n"
            "See the full stdout/stderr above for details."
        )
    return result


def install_environment() -> bool:
    spec_text = json.dumps(ENVIRONMENT_SPEC, sort_keys=True)
    if INSTALL_SENTINEL.exists() and INSTALL_SENTINEL.read_text(encoding="utf-8") == spec_text and not loaded_before_install:
        print("Dependency manifest already satisfied in this runtime; reusing the existing environment.")
        return False

    run_install(["apt-get", "update"])
    run_install(["apt-get", "install", "-y", *ENVIRONMENT_SPEC["apt"]])
    run_install([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
    run_install([sys.executable, "-m", "pip", "install", "--no-cache-dir", *ENVIRONMENT_SPEC["pip"]])

    repo_dir = Path(ENVIRONMENT_SPEC["nnunet_v1"]["path"])
    if repo_dir.exists() and not (repo_dir / ".git").exists():
        shutil.rmtree(repo_dir)
    if not repo_dir.exists():
        run_install(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                ENVIRONMENT_SPEC["nnunet_v1"]["branch"],
                ENVIRONMENT_SPEC["nnunet_v1"]["repo"],
                str(repo_dir),
            ]
        )

    setup_py = repo_dir / "setup.py"
    setup_text = setup_py.read_text(encoding="utf-8")
    patched_text = setup_text.replace('"sklearn"', '"scikit-learn"').replace("'sklearn'", "'scikit-learn'")
    if patched_text != setup_text:
        setup_py.write_text(patched_text, encoding="utf-8")
        print("Patched nnU-Net v1 setup.py: sklearn -> scikit-learn")
    else:
        print("nnU-Net v1 setup.py did not require a sklearn patch.")

    run_install([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(repo_dir)])

    INSTALL_SENTINEL.write_text(spec_text, encoding="utf-8")
    return True


environment_changed = install_environment()
print("\nDependency installation finished.")

if loaded_before_install and environment_changed:
    raise SystemExit(
        "Dependencies were reconciled after scientific/imaging packages were already imported in this runtime. "
        "Restart the Colab runtime now, then rerun from the next verification cell."
    )

print("Environment is coherent. Continue to the verification cell.")



$ apt-get update
[exit code] 0

$ apt-get install -y git git-lfs pigz p7zip-full unzip libgl1 libglib2.0-0
[exit code] 0

$ /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
[exit code] 0

$ /usr/bin/python3 -m pip install --no-cache-dir numpy==1.26.2 scipy==1.13.1 pandas==2.2.2 matplotlib==3.8.4 scikit-learn==1.5.2 tqdm==4.66.4 requests==2.32.3 reportlab==4.2.2 tabulate==0.9.0 SimpleITK==2.3.1 nibabel==5.2.1 pynrrd==1.0.0 pydicom==2.4.4 pybase64==1.3.2 httplib2==0.22.0 huggingface_hub==0.23.4 monai==1.4.0 pytorch-ignite==0.4.11 fire==0.6.0 MedProIO==0.1.1 batchgenerators==0.25 scikit-image==0.24.0 tifffile==2024.8.30 medpy==0.5.2 dicom2nifti==2.4.11 PyYAML==6.0.2 nnunetv2==2.2.1 'udocker>=1.3.17'
[exit code] 0

$ git clone --depth 1 --branch v1.7.1 https://github.com/MIC-DKFZ/nnUNet.git /content/nnUNet_v1
[exit code] 0
nnU-Net v1 setup.py did not require a sklearn patch.

$ /usr/bin/python3 -m pip install --no-deps -e /content/nnUNet_v1
[exit code] 0

Dependency installa

SystemExit: Dependencies were reconciled after scientific/imaging packages were already imported in this runtime. Restart the Colab runtime now, then rerun from the next verification cell.

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
import importlib
import importlib.util
import shutil
import sys
from pathlib import Path
from typing import Optional

import pandas as pd
from IPython.display import display


def describe_module(name: str, import_name: Optional[str] = None) -> dict[str, str]:
    import_name = import_name or name
    spec = importlib.util.find_spec(import_name)
    if spec is None:
        return {"package": name, "present": False, "version": "<missing>", "path": "<missing>"}
    module = importlib.import_module(import_name)
    location = getattr(module, "__file__", "<built-in>")
    version = getattr(module, "__version__", "unknown")
    return {
        "package": name,
        "present": True,
        "version": version,
        "path": str(Path(location).resolve()) if location != "<built-in>" else location,
    }


print(f"Python: {sys.version.replace(chr(10), ' ')}")
print(f"udocker executable: {shutil.which('udocker') or '<missing>'}")

display(
    pd.DataFrame(
        [
            describe_module("numpy"),
            describe_module("scipy"),
            describe_module("pandas"),
            describe_module("matplotlib"),
            describe_module("SimpleITK"),
            describe_module("nibabel"),
            describe_module("nrrd"),
            describe_module("pydicom"),
            describe_module("monai"),
            describe_module("ignite"),
            describe_module("fire"),
            describe_module("MedProIO"),
            describe_module("nnunetv2"),
        ]
    )
)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
udocker executable: /usr/local/bin/udocker


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


,package,present,version,path
0,numpy,True,1.26.2,/usr/local/lib/python3.12/dist-packages/numpy/...
1,scipy,True,1.13.1,/usr/local/lib/python3.12/dist-packages/scipy/...
2,pandas,True,2.2.2,/usr/local/lib/python3.12/dist-packages/pandas...
3,matplotlib,True,3.8.4,/usr/local/lib/python3.12/dist-packages/matplo...
4,SimpleITK,True,2.3.1,/usr/local/lib/python3.12/dist-packages/Simple...
5,nibabel,True,5.2.1,/usr/local/lib/python3.12/dist-packages/nibabe...
6,nrrd,True,1.0.0,/usr/local/lib/python3.12/dist-packages/nrrd/_...
7,pydicom,True,2.4.4,/usr/local/lib/python3.12/dist-packages/pydico...
8,monai,True,1.4.0,/usr/local/lib/python3.12/dist-packages/monai/...
9,ignite,True,0.4.11,/usr/local/lib/python3.12/dist-packages/ignite...


In [2]:
from __future__ import annotations

import importlib.util
import json
import logging
import math
import os
import random
import re
import shutil
import shlex
import subprocess
import sys
import tarfile
import time
import traceback
import zipfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import requests
import SimpleITK as sitk
from huggingface_hub import snapshot_download
from IPython.display import display
from reportlab.lib import colors
from reportlab.lib.enums import TA_LEFT
from reportlab.lib.pagesizes import landscape, letter
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import Image as RLImage
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle
from tqdm.auto import tqdm

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

plt.style.use("default")
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s: %(message)s", datefmt="%H:%M:%S")

print(f"Global random seed set to {SEED}")


Global random seed set to 2026


## Runtime Configuration

This cell keeps the benchmark configuration in one place.

- `TEST_DATA_URL` is prefilled
- `GT_DATA_URL` is optional if the extracted dataset already contains segmentation files
- `GT_LOCAL_PATH` is an optional alternative for a pre-staged runtime path
- runtime folders live under `/content`
- `MAX_CASES` can be used for debugging dry runs and should remain `None` for the full benchmark


In [3]:
TEST_DATA_URL = "https://zenodo.org/records/8026660/files/test_data.zip?download=1"

GT_DATA_URL = ""  # Optional if the extracted dataset already contains *_segmentation files
GT_LOCAL_PATH = ""  # Optional if the extracted dataset already contains *_segmentation files
ALLOW_GT_EMBEDDED_IN_TEST_ARCHIVE = True

MAX_CASES: Optional[int] = None
FORCE_REDOWNLOAD = False
FORCE_REEXTRACT = False

ROOT_DIR = Path("/content/promise12_benchmark")
DOWNLOADS_DIR = ROOT_DIR / "downloads"
EXTRACT_DIR = ROOT_DIR / "extracted"
TEST_EXTRACT_DIR = EXTRACT_DIR / "test_data"
GT_EXTRACT_DIR = EXTRACT_DIR / "ground_truth"
MODELS_DIR = ROOT_DIR / "models"
INTERMEDIATE_DIR = ROOT_DIR / "intermediate"
OUTPUTS_DIR = ROOT_DIR / "outputs"
PREDICTIONS_DIR = OUTPUTS_DIR / "predictions"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"
REPORTS_DIR = ROOT_DIR / "reports"

for path in [
    ROOT_DIR,
    DOWNLOADS_DIR,
    EXTRACT_DIR,
    TEST_EXTRACT_DIR,
    GT_EXTRACT_DIR,
    MODELS_DIR,
    INTERMEDIATE_DIR,
    OUTPUTS_DIR,
    PREDICTIONS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    REPORTS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

display(
    pd.DataFrame(
        [
            {"key": "TEST_DATA_URL", "value": TEST_DATA_URL},
            {"key": "GT_DATA_URL", "value": GT_DATA_URL or "<empty>"},
            {"key": "GT_LOCAL_PATH", "value": GT_LOCAL_PATH or "<empty>"},
            {"key": "ALLOW_GT_EMBEDDED_IN_TEST_ARCHIVE", "value": ALLOW_GT_EMBEDDED_IN_TEST_ARCHIVE},
            {"key": "MAX_CASES", "value": MAX_CASES},
            {"key": "ROOT_DIR", "value": str(ROOT_DIR)},
            {"key": "MODELS_DIR", "value": str(MODELS_DIR)},
            {"key": "OUTPUTS_DIR", "value": str(OUTPUTS_DIR)},
            {"key": "REPORTS_DIR", "value": str(REPORTS_DIR)},
        ]
    )
)


,key,value
0,TEST_DATA_URL,https://zenodo.org/records/8026660/files/test_...
1,GT_DATA_URL,<empty>
2,GT_LOCAL_PATH,<empty>
3,ALLOW_GT_EMBEDDED_IN_TEST_ARCHIVE,True
4,MAX_CASES,None
5,ROOT_DIR,/content/promise12_benchmark
6,MODELS_DIR,/content/promise12_benchmark/models
7,OUTPUTS_DIR,/content/promise12_benchmark/outputs
8,REPORTS_DIR,/content/promise12_benchmark/reports


## Download, Extraction, and Dataset Inspection

The benchmark should not depend on a brittle archive layout. The next cells:

- download archives with progress bars
- extract into dedicated runtime folders
- support either URL/local GT input or segmentation files embedded in the same dataset tree
- recursively discover medical images without hardcoding folder names
- support NIfTI, NRRD, and MetaImage headers (`.mhd`, `.mha`)


In [4]:
MEDICAL_EXTENSIONS = (".nii", ".nii.gz", ".nrrd", ".mhd", ".mha")
MASK_HINTS = {"mask", "masks", "label", "labels", "seg", "segmentation", "truth", "groundtruth", "ground_truth", "annotation", "annotations", "gt"}


def is_medical_image_path(path: Path) -> bool:
    name = path.name.lower()
    return any(name.endswith(ext) for ext in MEDICAL_EXTENSIONS)


def strip_medical_extension(name: str) -> str:
    for ext in [".nii.gz", ".nrrd", ".mhd", ".mha", ".nii"]:
        if name.lower().endswith(ext):
            return name[: -len(ext)]
    return name


def is_segmentation_like_path(path: Path) -> bool:
    lowered = strip_medical_extension(path.name).lower()
    return any(
        token in lowered
        for token in [
            "_seg",
            "segmentation",
            "_mask",
            "mask",
            "_label",
            "label",
            "_gt",
            "groundtruth",
            "ground_truth",
            "annotation",
        ]
    )


def download_file(url: str, destination: Path, force: bool = False) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not force:
        print(f"Reusing existing download: {destination}")
        return destination
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))
    with open(destination, "wb") as handle, tqdm(total=total, unit="B", unit_scale=True, unit_divisor=1024, desc=destination.name) as progress:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if not chunk:
                continue
            handle.write(chunk)
            progress.update(len(chunk))
    return destination


def extract_archive(archive_path: Path, destination_dir: Path, force: bool = False) -> Path:
    destination_dir.mkdir(parents=True, exist_ok=True)
    marker = destination_dir / ".extraction_complete"
    if marker.exists() and not force:
        print(f"Reusing existing extraction: {destination_dir}")
        return destination_dir
    if force:
        for child in destination_dir.iterdir():
            if child.is_dir():
                shutil.rmtree(child)
            else:
                child.unlink()
    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(destination_dir)
    elif tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path, "r:*") as tf:
            tf.extractall(destination_dir)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")
    marker.write_text("ok\n", encoding="utf-8")
    return destination_dir


def materialize_source(*, url: str, local_path: str, default_download_name: str, extraction_dir: Path, force_download: bool = False, force_extract: bool = False) -> Path:
    if url:
        archive_path = download_file(url, DOWNLOADS_DIR / default_download_name, force=force_download)
        return extract_archive(archive_path, extraction_dir, force=force_extract)
    if local_path:
        local = Path(local_path)
        if not local.exists():
            raise FileNotFoundError(f"Configured local path does not exist: {local}")
        if local.is_dir():
            return local
        return extract_archive(local, extraction_dir, force=force_extract)
    raise ValueError("A source URL or local path must be supplied.")


def discover_medical_images(root: Path) -> list[Path]:
    files = sorted(p for p in root.rglob("*") if p.is_file() and is_medical_image_path(p))
    if not files:
        raise FileNotFoundError(f"No supported medical image files were found under: {root}")
    return files


def pretty_case_name(path: Path) -> str:
    stem = strip_medical_extension(path.name)
    return re.sub(r"_0000$", "", stem, flags=re.IGNORECASE)


def canonical_case_key(path: Path) -> str:
    stem = pretty_case_name(path).lower()
    stem = re.sub(r"(ground[_\- ]?truth|segmentation|annotation|labels?|masks?|truth|_gt$|gt$|_seg$|seg$)", "", stem)
    stem = re.sub(r"[^a-z0-9]+", "", stem)
    return stem


def image_header_summary(path: Path) -> dict[str, Any]:
    image = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(image)
    return {
        "path": str(path),
        "size_xyz": tuple(int(v) for v in image.GetSize()),
        "spacing_xyz": tuple(float(v) for v in image.GetSpacing()),
        "origin_xyz": tuple(float(v) for v in image.GetOrigin()),
        "dtype": str(array.dtype),
        "min": float(array.min()),
        "max": float(array.max()),
        "nonzero_voxels": int(np.count_nonzero(array)),
    }


def partition_mri_and_gt(paths: list[Path]) -> tuple[list[Path], list[Path]]:
    mri_paths = [path for path in paths if not is_segmentation_like_path(path)]
    gt_paths = [path for path in paths if is_segmentation_like_path(path)]
    return mri_paths, gt_paths


In [5]:
test_root = materialize_source(
    url=TEST_DATA_URL,
    local_path="",
    default_download_name="promise12_test_data.zip",
    extraction_dir=TEST_EXTRACT_DIR,
    force_download=FORCE_REDOWNLOAD,
    force_extract=FORCE_REEXTRACT,
)

gt_root = None
if GT_DATA_URL or GT_LOCAL_PATH:
    gt_root = materialize_source(
        url=GT_DATA_URL,
        local_path=GT_LOCAL_PATH,
        default_download_name="promise12_ground_truth.zip",
        extraction_dir=GT_EXTRACT_DIR,
        force_download=FORCE_REDOWNLOAD,
        force_extract=FORCE_REEXTRACT,
    )

test_candidates = discover_medical_images(test_root)
mri_files, embedded_gt_files = partition_mri_and_gt(test_candidates)

gt_files: list[Path] = []
gt_source_label = "none"

if gt_root is not None:
    gt_candidates = discover_medical_images(gt_root)
    gt_files = gt_candidates
    gt_source_label = "external_gt_source"
elif ALLOW_GT_EMBEDDED_IN_TEST_ARCHIVE and embedded_gt_files:
    gt_files = embedded_gt_files
    gt_source_label = "embedded_in_test_archive"
else:
    raise ValueError(
        "Ground-truth masks were not found. Either set GT_DATA_URL / GT_LOCAL_PATH or supply a dataset archive "
        "that already contains segmentation files such as Case00_segmentation.mhd beside the MRI volume."
    )

print(f"Discovered {len(mri_files)} MRI candidate files")
print(f"Discovered {len(gt_files)} GT candidate files")
print(f"GT source mode: {gt_source_label}")
display(
    pd.DataFrame(
        {
            "role": ["MRI"] * len(mri_files) + ["GT"] * len(gt_files),
            "path": [str(p) for p in mri_files] + [str(p) for p in gt_files],
            "case_key": [canonical_case_key(p) for p in mri_files] + [canonical_case_key(p) for p in gt_files],
        }
    ).head(20)
)

print("\nRepresentative MRI headers:")
for path in mri_files[: min(2, len(mri_files))]:
    print(json.dumps(image_header_summary(path), indent=2))

print("\nRepresentative GT headers:")
for path in gt_files[: min(2, len(gt_files))]:
    print(json.dumps(image_header_summary(path), indent=2))


promise12_test_data.zip:   0%|          | 0.00/186M [00:00<?, ?B/s]

Discovered 30 MRI candidate files
Discovered 30 GT candidate files
GT source mode: embedded_in_test_archive


,role,path,case_key
0,MRI,/content/promise12_benchmark/extracted/test_da...,case00
1,MRI,/content/promise12_benchmark/extracted/test_da...,case01
2,MRI,/content/promise12_benchmark/extracted/test_da...,case02
3,MRI,/content/promise12_benchmark/extracted/test_da...,case03
4,MRI,/content/promise12_benchmark/extracted/test_da...,case04
5,MRI,/content/promise12_benchmark/extracted/test_da...,case05
6,MRI,/content/promise12_benchmark/extracted/test_da...,case06
7,MRI,/content/promise12_benchmark/extracted/test_da...,case07
8,MRI,/content/promise12_benchmark/extracted/test_da...,case08
9,MRI,/content/promise12_benchmark/extracted/test_da...,case09



Representative MRI headers:
{
  "path": "/content/promise12_benchmark/extracted/test_data/Case00.mhd",
  "size_xyz": [
    320,
    320,
    20
  ],
  "spacing_xyz": [
    0.625,
    0.625,
    3.60001
  ],
  "origin_xyz": [
    -112.807,
    -114.013,
    -86.4137
  ],
  "dtype": "int16",
  "min": 0.0,
  "max": 1831.0,
  "nonzero_voxels": 2035299
}
{
  "path": "/content/promise12_benchmark/extracted/test_data/Case01.mhd",
  "size_xyz": [
    320,
    320,
    20
  ],
  "spacing_xyz": [
    0.625,
    0.625,
    3.6
  ],
  "origin_xyz": [
    -118.618,
    -94.4006,
    -123.097
  ],
  "dtype": "int16",
  "min": 0.0,
  "max": 1385.0,
  "nonzero_voxels": 2034687
}

Representative GT headers:
{
  "path": "/content/promise12_benchmark/extracted/test_data/Case00_segmentation.mhd",
  "size_xyz": [
    320,
    320,
    20
  ],
  "spacing_xyz": [
    0.625,
    0.625,
    3.60001
  ],
  "origin_xyz": [
    -112.807,
    -114.013,
    -86.4137
  ],
  "dtype": "int8",
  "min": 0.0,
  "max": 1

## Case Pairing

The notebook pairs MRI and GT files programmatically and refuses to guess if the mapping is ambiguous.

Pairing policy:

- normalize file names into canonical case keys
- remove common mask-specific suffixes
- support PROMISE12-style names such as `Case00.mhd` and `Case00_segmentation.mhd`
- require a strict one-to-one MRI/GT match
- raise explicit diagnostics for duplicates or unmatched keys


In [6]:
@dataclass
class CaseItem:
    case_id: str
    case_key: str
    mri_path: Path
    gt_path: Path


def build_unique_index(paths: list[Path], role: str) -> dict[str, Path]:
    grouped: dict[str, list[Path]] = {}
    for path in paths:
        grouped.setdefault(canonical_case_key(path), []).append(path)
    duplicates = {key: values for key, values in grouped.items() if len(values) > 1}
    if duplicates:
        details = "\n".join(
            f"{role} key '{key}' has {len(values)} candidates:\n  " + "\n  ".join(str(v) for v in values)
            for key, values in sorted(duplicates.items())
        )
        raise ValueError(f"Ambiguous {role} discovery. Duplicate normalized keys were found.\n{details}")
    return {key: values[0] for key, values in grouped.items()}


def pair_cases(mri_paths: list[Path], gt_paths: list[Path]) -> list[CaseItem]:
    mri_index = build_unique_index(mri_paths, "MRI")
    gt_index = build_unique_index(gt_paths, "GT")
    only_mri = sorted(set(mri_index) - set(gt_index))
    only_gt = sorted(set(gt_index) - set(mri_index))
    if only_mri or only_gt:
        diagnostics = []
        if only_mri:
            diagnostics.append("MRI-only keys:\n  " + "\n  ".join(only_mri))
        if only_gt:
            diagnostics.append("GT-only keys:\n  " + "\n  ".join(only_gt))
        raise ValueError("MRI/GT pairing failed because the discovered case keys do not match unambiguously.\n" + "\n\n".join(diagnostics))
    return [
        CaseItem(case_id=pretty_case_name(mri_index[key]), case_key=key, mri_path=mri_index[key], gt_path=gt_index[key])
        for key in sorted(mri_index)
    ]


cases = pair_cases(mri_files, gt_files)
if MAX_CASES is not None:
    cases = cases[:MAX_CASES]

display(
    pd.DataFrame(
        [{"case_id": case.case_id, "case_key": case.case_key, "mri_path": str(case.mri_path), "gt_path": str(case.gt_path)} for case in cases]
    ).head(20)
)
print(f"Successfully paired {len(cases)} cases.")


,case_id,case_key,mri_path,gt_path
0,Case00,case00,/content/promise12_benchmark/extracted/test_da...,/content/promise12_benchmark/extracted/test_da...
1,Case01,case01,/content/promise12_benchmark/extracted/test_da...,/content/promise12_benchmark/extracted/test_da...
2,Case02,case02,/content/promise12_benchmark/extracted/test_da...,/content/promise12_benchmark/extracted/test_da...
3,Case03,case03,/content/promise12_benchmark/extracted/test_da...,/content/promise12_benchmark/extracted/test_da...
4,Case04,case04,/content/promise12_benchmark/extracted/test_da...,/content/promise12_benchmark/extracted/test_da...
5,Case05,case05,/content/promise12_benchmark/extracted/test_da...,/content/promise12_benchmark/extracted/test_da...
6,Case06,case06,/content/promise12_benchmark/extracted/test_da...,/content/promise12_benchmark/extracted/test_da...
7,Case07,case07,/content/promise12_benchmark/extracted/test_da...,/content/promise12_benchmark/extracted/test_da...
8,Case08,case08,/content/promise12_benchmark/extracted/test_da...,/content/promise12_benchmark/extracted/test_da...
9,Case09,case09,/content/promise12_benchmark/extracted/test_da...,/content/promise12_benchmark/extracted/test_da...


Successfully paired 30 cases.


## Common Geometry and File Utilities

Before any metric is computed, every model output must be converted into the same final representation:

- binary mask only
- label set `{0, 1}`
- same voxel grid as the ground truth
- same spatial reference as the ground truth
- same anatomical target: whole gland

The utilities below handle format conversion, geometry validation, optional resampling, and consistent saving. All four model adapters use the same standardization path.


In [7]:
def safe_slug(text: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")
    return slug or "model"


def ensure_clean_dir(path: Path) -> Path:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def load_image(path: Path) -> sitk.Image:
    return sitk.ReadImage(str(path))


def save_image(image: sitk.Image, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    sitk.WriteImage(image, str(path))
    return path


def convert_image_format(source_path: Path, destination_path: Path) -> Path:
    image = load_image(source_path)
    return save_image(image, destination_path)


def image_geometry_signature(image: sitk.Image) -> dict[str, Any]:
    return {
        "size": tuple(int(v) for v in image.GetSize()),
        "spacing": tuple(float(v) for v in image.GetSpacing()),
        "origin": tuple(float(v) for v in image.GetOrigin()),
        "direction": tuple(float(v) for v in image.GetDirection()),
    }


def same_geometry(a: sitk.Image, b: sitk.Image, tol: float = 1e-5) -> bool:
    if a.GetSize() != b.GetSize():
        return False
    for av, bv in zip(a.GetSpacing(), b.GetSpacing()):
        if abs(float(av) - float(bv)) > tol:
            return False
    for av, bv in zip(a.GetOrigin(), b.GetOrigin()):
        if abs(float(av) - float(bv)) > tol:
            return False
    for av, bv in zip(a.GetDirection(), b.GetDirection()):
        if abs(float(av) - float(bv)) > tol:
            return False
    return True


def binary_image_from_labels(source_image: sitk.Image, *, positive_labels: Optional[set[int]] = None) -> sitk.Image:
    array = sitk.GetArrayFromImage(source_image)
    if positive_labels is None:
        binary = (array > 0).astype(np.uint8)
    else:
        binary = np.isin(array, list(positive_labels)).astype(np.uint8)
    image = sitk.GetImageFromArray(binary)
    image.CopyInformation(source_image)
    return image


def resample_image_to_reference(image: sitk.Image, reference: sitk.Image, *, interpolator: int, default_value: float = 0.0) -> sitk.Image:
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(reference)
    resampler.SetInterpolator(interpolator)
    resampler.SetDefaultPixelValue(default_value)
    return resampler.Execute(image)


def standardize_prediction_to_gt_space(raw_prediction_path: Path, gt_path: Path, standardized_output_path: Path, *, positive_labels: Optional[set[int]] = None) -> dict[str, Any]:
    gt_image = load_image(gt_path)
    raw_prediction_image = load_image(raw_prediction_path)
    warnings_list: list[str] = []
    binary_prediction = binary_image_from_labels(raw_prediction_image, positive_labels=positive_labels)

    if not same_geometry(binary_prediction, gt_image):
        warnings_list.append("prediction_resampled_to_gt_space")
        binary_prediction = resample_image_to_reference(
            binary_prediction,
            gt_image,
            interpolator=sitk.sitkNearestNeighbor,
            default_value=0,
        )
        binary_prediction = binary_image_from_labels(binary_prediction, positive_labels={1})

    if binary_prediction.GetSize() != gt_image.GetSize():
        raise ValueError(
            "Prediction could not be aligned safely to the GT voxel grid. "
            f"Prediction size: {binary_prediction.GetSize()} | GT size: {gt_image.GetSize()}"
        )

    save_image(binary_prediction, standardized_output_path)
    return {
        "standardized_path": standardized_output_path,
        "warnings": warnings_list,
        "prediction_geometry": image_geometry_signature(binary_prediction),
        "gt_geometry": image_geometry_signature(gt_image),
        "prediction_positive_voxels": int(np.count_nonzero(sitk.GetArrayFromImage(binary_prediction))),
        "gt_positive_voxels": int(np.count_nonzero(sitk.GetArrayFromImage(binary_image_from_labels(gt_image, positive_labels=None)))),
    }


def align_image_for_display(image_path: Path, reference_path: Path) -> sitk.Image:
    image = load_image(image_path)
    reference = load_image(reference_path)
    if same_geometry(image, reference):
        return image
    return resample_image_to_reference(
        image,
        reference,
        interpolator=sitk.sitkLinear,
        default_value=float(np.min(sitk.GetArrayFromImage(image))),
    )


def locate_prediction_file(root: Path, suffixes: tuple[str, ...] = (".nii.gz", ".nii", ".nrrd")) -> Path:
    candidates = sorted(p for p in root.rglob("*") if p.is_file() and any(p.name.lower().endswith(s) for s in suffixes))
    if not candidates:
        raise FileNotFoundError(f"No prediction file was found under {root}")
    return candidates[0]


def load_python_module(module_path: Path, module_name: str):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    assert spec is not None and spec.loader is not None
    spec.loader.exec_module(module)
    return module


## Metric Definitions

The benchmark uses exactly three metrics on the final standardized **binary whole-gland mask** in GT space.

### Dice
`Dice = 2TP / (2TP + FP + FN)`  
Higher is better.

### Tversky
`Tversky = TP / (TP + alpha*FP + beta*FN)` with:

- `alpha = 0.3`
- `beta = 0.7`

Higher is better.

### Non-Intersecting Prediction Percentage
`NonIntersectPct = 100 * FP / (TP + FP)`  
Lower is better, and a value of `0` is desirable.

### Edge-Case Convention

- if prediction is empty and GT is non-empty:
  - Dice = 0
  - Tversky = 0
  - NonIntersectPct = 0
- if both prediction and GT are empty:
  - Dice = 1
  - Tversky = 1
  - NonIntersectPct = 0


In [8]:
TVERSKY_ALPHA = 0.3
TVERSKY_BETA = 0.7


def compute_binary_metrics(prediction_path: Path, gt_path: Path) -> dict[str, Any]:
    prediction = sitk.GetArrayFromImage(load_image(prediction_path)).astype(bool)
    ground_truth = sitk.GetArrayFromImage(load_image(gt_path)).astype(bool)

    if prediction.shape != ground_truth.shape:
        raise ValueError(f"Metric computation received mismatched arrays: pred={prediction.shape}, gt={ground_truth.shape}")

    tp = int(np.logical_and(prediction, ground_truth).sum())
    fp = int(np.logical_and(prediction, np.logical_not(ground_truth)).sum())
    fn = int(np.logical_and(np.logical_not(prediction), ground_truth).sum())

    pred_sum = int(prediction.sum())
    gt_sum = int(ground_truth.sum())
    warnings_list: list[str] = []

    if pred_sum == 0 and gt_sum > 0:
        warnings_list.append("empty_prediction_nonempty_gt")
        return {"dice": 0.0, "tversky": 0.0, "non_intersect_pct": 0.0, "tp": tp, "fp": fp, "fn": fn, "warnings": warnings_list}

    if pred_sum == 0 and gt_sum == 0:
        warnings_list.append("both_prediction_and_gt_empty")
        return {"dice": 1.0, "tversky": 1.0, "non_intersect_pct": 0.0, "tp": tp, "fp": fp, "fn": fn, "warnings": warnings_list}

    dice = (2.0 * tp) / max((2 * tp + fp + fn), 1)
    tversky = tp / max((tp + TVERSKY_ALPHA * fp + TVERSKY_BETA * fn), 1e-8)
    non_intersect_pct = 100.0 * fp / max((tp + fp), 1)

    return {
        "dice": float(dice),
        "tversky": float(tversky),
        "non_intersect_pct": float(non_intersect_pct),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "warnings": warnings_list,
    }


## Adapter Interface

Each model is wrapped behind the same public contract:

- `setup()`
- `predict_case(case)`

The adapter is responsible for:

- obtaining its own repo / bundle / weights at runtime
- handling any model-specific input formatting
- running inference
- returning a raw prediction
- standardizing the final scored mask into GT space


In [9]:
class ModelBlockedError(RuntimeError):
    pass


@dataclass
class PredictionResult:
    model_name: str
    case_id: str
    raw_prediction_path: Optional[Path] = None
    standardized_prediction_path: Optional[Path] = None
    runtime_seconds: Optional[float] = None
    warnings: list[str] = field(default_factory=list)
    failure_reason: Optional[str] = None


class BaseModelAdapter:
    model_name = "base"

    def __init__(self, work_dir: Path):
        self.work_dir = work_dir
        self.work_dir.mkdir(parents=True, exist_ok=True)
        self.ready = False
        self.setup_error: Optional[str] = None
        self.setup_notes: list[str] = []

    def setup(self) -> None:
        raise NotImplementedError

    def predict_case(self, case: CaseItem) -> PredictionResult:
        raise NotImplementedError

    def ensure_ready(self):
        if not self.ready:
            raise ModelBlockedError(self.setup_error or f"{self.model_name} is not ready.")


def run_command(cmd: list[str], *, cwd: Optional[Path] = None, env: Optional[dict[str, str]] = None, check: bool = True) -> str:
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"$ {printable}")
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        lines.append(line)
    process.wait()
    output = "".join(lines)
    if check and process.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {process.returncode}: {printable}\n{output[-4000:]}")
    return output


def resolve_case_output_dir(model_name: str, case_id: str) -> Path:
    return PREDICTIONS_DIR / safe_slug(model_name) / safe_slug(case_id)


def recursive_find_first_checkpoint(repo_dir: Path) -> list[Path]:
    return sorted(repo_dir.rglob("*.pth"))


def flatten_warning_list(*warning_groups: list[str]) -> list[str]:
    combined: list[str] = []
    for group in warning_groups:
        combined.extend(group)
    return sorted(dict.fromkeys(item for item in combined if item))


In [10]:
import importlib.util
import numpy as np


required_modules = {
    "pydicom": "pydicom",
    "nnunetv2": "nnunetv2",
    "MedProIO": "MedProIO",
}
missing_modules = [label for label, import_name in required_modules.items() if importlib.util.find_spec(import_name) is None]
if missing_modules:
    raise RuntimeError(
        "The Dzaridis runtime dependencies are missing from the unified environment cell: "
        + ", ".join(sorted(missing_modules))
    )

if np.__version__ != "1.26.2":
    raise RuntimeError(
        f"NumPy drift detected ({np.__version__}). The notebook is pinned to numpy==1.26.2 in the main install cell. "
        "Restart the runtime and rerun from the top instead of reinstalling NumPy mid-session."
    )

print("Dzaridis dependencies are already provided by the unified environment cell.")
print(f"NumPy version: {np.__version__}")


Dzaridis dependencies are already provided by the unified environment cell.
NumPy version: 1.26.2


In [19]:
import contextlib
import torch

@contextlib.contextmanager
def legacy_torch_load_context():
    """
    Temporarily restore old torch.load behavior for legacy checkpoints
    that break under PyTorch>=2.6 default weights_only=True.
    Use only for trusted local model checkpoints.
    """
    original_torch_load = torch.load

    def patched_torch_load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return original_torch_load(*args, **kwargs)

    torch.load = patched_torch_load
    try:
        yield
    finally:
        torch.load = original_torch_load

## Model 1: Dzaridis Adapter

Colab route:

- clone the official repository
- create the expected `Pats` and `Outputs` folders automatically
- run the repository pipeline case by case
- use the repository's **resampled-to-original-space** whole-gland binary output for scoring

The whole-gland output used for evaluation is the `wg_binary` path recorded in the repository's `ResampledToOriginalSegmentationPaths.json`.


In [20]:
def ensure_numpy_char_compat() -> None:
    """
    NumPy 1.x exposes np.char, but `import numpy.char` can fail because
    `numpy.char` is not an importable submodule in those releases.
    Some third-party packages still try to import it that way.
    """
    import importlib
    import types

    try:
        importlib.import_module("numpy.char")
        return
    except ModuleNotFoundError:
        pass

    char_source = getattr(np, "char", None)
    if char_source is None:
        raise RuntimeError("np.char is unavailable, so the numpy.char compatibility shim cannot be created.")

    shim = types.ModuleType("numpy.char")
    for name in dir(char_source):
        try:
            setattr(shim, name, getattr(char_source, name))
        except Exception:
            pass

    shim.__dict__.setdefault("__doc__", getattr(char_source, "__doc__", "Compatibility shim for numpy.char"))
    sys.modules["numpy.char"] = shim
    setattr(np, "char", shim)
    print("Installed numpy.char compatibility shim for NumPy 1.x.")


def normalize_dzaridis_key(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", strip_medical_extension(str(text)).lower())


def repo_relative_string(path: Path, repo_dir: Path) -> str:
    return os.path.relpath(path, repo_dir).replace("\\", "/")


def build_dzaridis_patient_list(pats_dir: Path, get_images_module) -> list[str]:
    import yaml

    dicom_files = sorted(pats_dir.rglob("*.dcm"))
    if dicom_files:
        return [str(p) for p in get_images_module.get_images("Pats")]

    nii_files = sorted(p for p in pats_dir.rglob("*.nii.gz") if p.is_file())
    if not nii_files:
        raise FileNotFoundError(f"No .nii.gz files were found under {pats_dir}")

    patient_dict = {
        str(path): {
            "destination_nifti": str(path),
            "source_type": "nii.gz",
        }
        for path in nii_files
    }
    with open(pats_dir / "patient_dict.yaml", "w", encoding="utf-8") as handle:
        yaml.safe_dump(patient_dict, handle, indent=4, sort_keys=False)
    return [str(path) for path in nii_files]


def persist_dzaridis_wg_mappings(repo_dir: Path, wg_dict_original: dict, wg_dict_resampled: dict) -> tuple[Path, Path]:
    outputs_dir = repo_dir / "Outputs"
    outputs_dir.mkdir(parents=True, exist_ok=True)

    def normalize_mapping(mapping: dict) -> dict:
        normalized: dict[str, dict[str, str]] = {}
        for key, values in mapping.items():
            normalized[key] = {}
            for label, raw_path in values.items():
                path = Path(raw_path)
                if not path.is_absolute():
                    path = repo_dir / path
                normalized[key][label] = repo_relative_string(path.resolve(), repo_dir)
        return normalized

    original_path = outputs_dir / "nnOutputSegmentationPaths.json"
    resampled_path = outputs_dir / "ResampledToOriginalSegmentationPaths.json"
    original_path.write_text(json.dumps(normalize_mapping(wg_dict_original), indent=4), encoding="utf-8")
    resampled_path.write_text(json.dumps(normalize_mapping(wg_dict_resampled), indent=4), encoding="utf-8")
    return original_path, resampled_path


def recover_dzaridis_wg_mappings(repo_dir: Path) -> tuple[dict, dict]:
    outputs_dir = repo_dir / "Outputs"
    original: dict[str, dict[str, str]] = {}
    resampled: dict[str, dict[str, str]] = {}
    if not outputs_dir.exists():
        return original, resampled

    for case_dir in sorted(p for p in outputs_dir.iterdir() if p.is_dir()):
        original_paths = {}
        resampled_paths = {}
        for label in ["wg_binary", "wg_probs"]:
            original_candidate = case_dir / "Original" / f"{label}.nii.gz"
            resampled_candidate = case_dir / "Resampled" / f"{label}.nii.gz"
            if original_candidate.exists():
                original_paths[label] = repo_relative_string(original_candidate.resolve(), repo_dir)
            if resampled_candidate.exists():
                resampled_paths[label] = repo_relative_string(resampled_candidate.resolve(), repo_dir)
        if original_paths:
            original[case_dir.name] = original_paths
        if resampled_paths:
            resampled[case_dir.name] = resampled_paths
    return original, resampled


def create_dzaridis_wg_outputs(
    repo_dir: Path,
    pats: dict,
    pats_for_wg_inference: dict,
    image_processor_module,
) -> tuple[dict, dict, list[str]]:
    warnings_list: list[str] = []
    wg_dict_original: dict[str, dict[str, str]] = {}
    wg_dict_resampled: dict[str, dict[str, str]] = {}

    for key, inference_paths in pats_for_wg_inference.items():
        binary_path = Path(inference_paths["binary"])
        if not binary_path.exists():
            raise FileNotFoundError(f"Dzaridis whole-gland output is missing: {binary_path}")

        wg_binary = sitk.ReadImage(str(binary_path))
        try:
            wg_binary = image_processor_module.process_mask(wg_binary)
            wg_binary = image_processor_module.remove_small_components(wg_binary)
        except Exception as exc:
            warnings_list.append(f"wg_binary_cleanup_warning:{type(exc).__name__}")

        case_output_dir = repo_dir / "Outputs" / key
        original_dir = case_output_dir / "Original"
        resampled_dir = case_output_dir / "Resampled"
        original_dir.mkdir(parents=True, exist_ok=True)
        resampled_dir.mkdir(parents=True, exist_ok=True)

        original_binary_path = original_dir / "wg_binary.nii.gz"
        resampled_binary_path = resampled_dir / "wg_binary.nii.gz"

        sitk.WriteImage(wg_binary, str(original_binary_path))
        sitk.WriteImage(
            sitk.Resample(wg_binary, pats[key], sitk.Transform(), sitk.sitkNearestNeighbor),
            str(resampled_binary_path),
        )

        original_mapping = {"wg_binary": repo_relative_string(original_binary_path.resolve(), repo_dir)}
        resampled_mapping = {"wg_binary": repo_relative_string(resampled_binary_path.resolve(), repo_dir)}

        probs_value = inference_paths.get("probs")
        if probs_value:
            probs_path = Path(probs_value)
        else:
            probs_path = None
        if probs_path is not None and probs_path.exists():
            try:
                probabilities = np.load(probs_path)["probabilities"][1]
                wg_probs = sitk.GetImageFromArray(probabilities)
                wg_probs.CopyInformation(wg_binary)

                original_probs_path = original_dir / "wg_probs.nii.gz"
                resampled_probs_path = resampled_dir / "wg_probs.nii.gz"

                sitk.WriteImage(wg_probs, str(original_probs_path))
                sitk.WriteImage(
                    sitk.Resample(wg_probs, pats[key], sitk.Transform(), sitk.sitkNearestNeighbor),
                    str(resampled_probs_path),
                )

                original_mapping["wg_probs"] = repo_relative_string(original_probs_path.resolve(), repo_dir)
                resampled_mapping["wg_probs"] = repo_relative_string(resampled_probs_path.resolve(), repo_dir)
            except Exception as exc:
                warnings_list.append(f"wg_probability_export_warning:{type(exc).__name__}")

        wg_dict_original[key] = original_mapping
        wg_dict_resampled[key] = resampled_mapping

    return wg_dict_original, wg_dict_resampled, warnings_list


def resolve_dzaridis_mapping_path(repo_dir: Path, mapping: dict, case: CaseItem) -> Path:
    target = normalize_dzaridis_key(case.case_id)
    for key, values in mapping.items():
        if normalize_dzaridis_key(key) != target:
            continue
        candidate = values.get("wg_binary")
        if not candidate:
            break
        path = Path(candidate)
        if not path.is_absolute():
            path = repo_dir / path
        return path
    raise KeyError(
        f"Could not resolve the Dzaridis whole-gland output for case '{case.case_id}'. "
        f"Available keys: {list(mapping.keys())[:10]}"
    )


class DzaridisAdapter(BaseModelAdapter):
    model_name = "Dzaridis"

    def __init__(self, work_dir: Path):
        super().__init__(work_dir)
        self.repo_dir = self.work_dir / "MRI-Prostate-Gland-and-Zone-Segmentor"
        self.segmentor_pipeline_module = None
        self.inputcheck_module = None
        self.get_images_module = None
        self.image_processor_module = None

    def setup(self) -> None:
        try:
            if not self.repo_dir.exists():
                run_command(
                    [
                        "git",
                        "clone",
                        "--depth",
                        "1",
                        "https://github.com/dzaridis/MRI-Prostate-Gland-and-Zone-Segmentor.git",
                        str(self.repo_dir),
                    ]
                )

            run_command(["git", "lfs", "install"], cwd=self.repo_dir, check=False)
            run_command(["git", "lfs", "pull"], cwd=self.repo_dir, check=False)

            for folder_name in ["Pats", "Outputs", "dicom_outputs"]:
                (self.repo_dir / folder_name).mkdir(parents=True, exist_ok=True)

            if str(self.repo_dir) not in sys.path:
                sys.path.insert(0, str(self.repo_dir))

            ensure_numpy_char_compat()
            self.segmentor_pipeline_module = load_python_module(
                self.repo_dir / "Utils" / "segmentor_pipeline.py",
                "dzaridis_segmentor_pipeline",
            )
            self.inputcheck_module = load_python_module(
                self.repo_dir / "Utils" / "InputCheck.py",
                "dzaridis_inputcheck",
            )
            self.get_images_module = load_python_module(
                self.repo_dir / "Utils" / "get_images.py",
                "dzaridis_get_images",
            )
            self.image_processor_module = load_python_module(
                self.repo_dir / "Utils" / "ImageProcessor.py",
                "dzaridis_image_processor",
            )

            checkpoints = recursive_find_first_checkpoint(self.repo_dir)
            if not checkpoints:
                raise RuntimeError(
                    "No .pth checkpoint files were found after cloning the Dzaridis repository. "
                    "This usually means the pretrained assets were not fetched correctly."
                )

            self.setup_notes.append("NIfTI-only Colab path enabled; optional DICOM/Orthanc export is bypassed.")
            self.setup_notes.append("Whole-gland benchmark output is generated directly from the repo's WG stage.")
            self.setup_notes.append(f"Discovered {len(checkpoints)} checkpoint file(s).")
            self.ready = True
        except Exception as exc:
            self.setup_error = f"{type(exc).__name__}: {exc}"
            raise

    def _clean_repo_runtime_dirs(self) -> None:
        runtime_paths = [
            self.repo_dir / "Pats",
            self.repo_dir / "Outputs",
            self.repo_dir / "dicom_outputs",
            self.repo_dir / "nnUnet_paths" / "nnUNet_raw" / "Dataset016_WgSegmentationPNetAndPicai" / "ImagesTs",
            self.repo_dir / "nnUnet_paths" / "nnUNet_raw" / "Dataset019_ProstateZonesSegmentationWgFilteredLessDilated" / "ImagesTs",
            self.repo_dir / "nnUnet_paths" / "nnUNet_raw" / "OutcomesWG",
            self.repo_dir / "nnUnet_paths" / "nnUNet_raw" / "OutcomesZones",
        ]
        for folder in runtime_paths:
            if folder.exists():
                shutil.rmtree(folder)
            if folder.name in {"Pats", "Outputs", "dicom_outputs", "ImagesTs"}:
                folder.mkdir(parents=True, exist_ok=True)

    def predict_case(self, case: CaseItem) -> PredictionResult:
        self.ensure_ready()
        self._clean_repo_runtime_dirs()

        result = PredictionResult(model_name=self.model_name, case_id=case.case_id)
        case_warnings: list[str] = []
        repo_case_input = self.repo_dir / "Pats" / f"{case.case_id}.nii.gz"
        convert_image_format(case.mri_path, repo_case_input)

        start = time.time()
        original_cwd = Path.cwd()
        try:
            os.chdir(self.repo_dir)
            patient_list = build_dzaridis_patient_list(self.repo_dir / "Pats", self.get_images_module)
            pats = self.inputcheck_module.load_nii_gz_files(patient_list)
            if not pats:
                raise RuntimeError("Dzaridis input loading returned no patients.")

            segmentor = self.segmentor_pipeline_module.Segmentor()
            with legacy_torch_load_context():
                  segmentor.wg_model(pats)

            if not segmentor.pats_for_wg_inference:
                raise RuntimeError("Dzaridis whole-gland inference produced no output path dictionary.")

            missing_raw_outputs = [
                info["binary"]
                for info in segmentor.pats_for_wg_inference.values()
                if not Path(info["binary"]).exists()
            ]
            if missing_raw_outputs:
                raise FileNotFoundError(
                    "Dzaridis whole-gland inference finished without writing the expected raw NIfTI outputs: "
                    + ", ".join(str(path) for path in missing_raw_outputs)
                )

            try:
                segmentor.preparation_zones(input_patients=pats)
            except Exception as exc:
                case_warnings.append(f"wg_output_rebuild_triggered:{type(exc).__name__}")

            wg_dict_original = segmentor.wg_dict_original or {}
            wg_dict_resampled = segmentor.wg_dict_resampled or {}

            if not wg_dict_original or not wg_dict_resampled:
                rebuilt_original, rebuilt_resampled, rebuild_warnings = create_dzaridis_wg_outputs(
                    self.repo_dir,
                    pats,
                    segmentor.pats_for_wg_inference,
                    self.image_processor_module,
                )
                wg_dict_original = rebuilt_original
                wg_dict_resampled = rebuilt_resampled
                case_warnings.extend(rebuild_warnings)
                case_warnings.append("wg_json_mapping_rebuilt_without_zone_stage")

            if not wg_dict_original or not wg_dict_resampled:
                recovered_original, recovered_resampled = recover_dzaridis_wg_mappings(self.repo_dir)
                wg_dict_original = wg_dict_original or recovered_original
                wg_dict_resampled = wg_dict_resampled or recovered_resampled
                if recovered_original or recovered_resampled:
                    case_warnings.append("wg_json_mapping_recovered_from_outputs")

            if not wg_dict_original or not wg_dict_resampled:
                raise RuntimeError(
                    "Dzaridis whole-gland inference completed, but the adapter could not build the required WG JSON mappings."
                )

            persist_dzaridis_wg_mappings(self.repo_dir, wg_dict_original, wg_dict_resampled)
        finally:
            os.chdir(original_cwd)

        result.runtime_seconds = time.time() - start

        mapping_path = self.repo_dir / "Outputs" / "ResampledToOriginalSegmentationPaths.json"
        if not mapping_path.exists():
            raise FileNotFoundError(
                "The Dzaridis adapter did not create Outputs/ResampledToOriginalSegmentationPaths.json."
            )

        mapping = json.loads(mapping_path.read_text(encoding="utf-8"))
        raw_prediction_path = resolve_dzaridis_mapping_path(self.repo_dir, mapping, case)
        if not raw_prediction_path.exists():
            raise FileNotFoundError(f"The resolved Dzaridis whole-gland output does not exist: {raw_prediction_path}")

        standardized_path = resolve_case_output_dir(self.model_name, case.case_id) / "prediction_binary_aligned.nii.gz"
        standardization = standardize_prediction_to_gt_space(
            raw_prediction_path,
            case.gt_path,
            standardized_path,
            positive_labels=None,
        )

        result.raw_prediction_path = raw_prediction_path
        result.standardized_prediction_path = standardization["standardized_path"]
        result.warnings = flatten_warning_list(self.setup_notes, case_warnings, standardization["warnings"])
        return result


## Model 2: DeepInfer Adapter

Colab route:

- use the official DeepInfer prostate segmenter
- keep the PROMISE12 domain
- choose `ProcessingType=Accurate`
- choose `Inference=Ensemble`
- convert inputs to `.nrrd`
- attempt container execution through `udocker --allow-root`

This is the most fragile hosted-Colab path in the notebook because the official distribution is Docker-centric.


In [12]:
class DeepInferAdapter(BaseModelAdapter):
    model_name = "DeepInfer"

    def __init__(self, work_dir: Path):
        super().__init__(work_dir)
        self.container_name = "deepinfer_prostate"
        self.image_ref_candidates = ["deepinfer/prostate:latest", "deepinfer/deepinfer-prostate:latest"]
        self.image_ref: Optional[str] = None
        self.fit_script_path: Optional[str] = None
        self.container_inspect_output = ""
        self.container_verify_output = ""
        self.container_probe_output = ""
        self.env = {**os.environ, "UDOCKER_DIR": str(self.work_dir / ".udocker")}

    def _udocker(self, *args: str, check: bool = True) -> str:
        return run_command(["udocker", "--allow-root", *args], env=self.env, check=check)

    def _resolve_image(self) -> str:
        last_error = None
        for candidate in self.image_ref_candidates:
            try:
                self._udocker("pull", candidate)
                return candidate
            except Exception as exc:
                last_error = exc
        raise RuntimeError(f"Unable to pull any supported DeepInfer image reference: {last_error}")

    def _discover_fit_script(self) -> str:
        try:
            probe_output = self._udocker(
                "run",
                "--workdir=/",
                "--entrypoint=/bin/sh",
                self.container_name,
                "-lc",
                "pwd; ls -la /deepinfer || true; find / -maxdepth 3 -path '*/fit.py' 2>/dev/null | sort | head -20",
                check=False,
            )
            self.container_probe_output = probe_output
            fit_candidates = [line.strip() for line in probe_output.splitlines() if line.strip().endswith("fit.py")]
            deepinfer_candidates = [path for path in fit_candidates if "/deepinfer/" in path or path == "/deepinfer/fit.py"]
            if deepinfer_candidates:
                return deepinfer_candidates[0]
            if fit_candidates:
                return fit_candidates[0]
        except Exception:
            pass

        if "deepinfer/fit.py" in self.container_inspect_output:
            return "/deepinfer/fit.py"

        raise RuntimeError(
            "Could not locate the DeepInfer runner inside the udocker container. "
            "Inspect output did not expose a usable fit.py path."
        )

    def setup(self) -> None:
        try:
            if not shutil.which("udocker"):
                raise RuntimeError("udocker is not installed in the current Colab runtime.")

            self._udocker("install")
            self.image_ref = self._resolve_image()

            ps_output = self._udocker("ps", check=False)
            if self.container_name not in ps_output:
                self._udocker("create", f"--name={self.container_name}", self.image_ref)

            self._udocker("setup", "--execmode=P1", self.container_name, check=False)
            self.container_inspect_output = self._udocker("inspect", self.container_name, check=False)
            self.container_verify_output = self._udocker("verify", self.container_name, check=False)
            self.fit_script_path = self._discover_fit_script()

            self.setup_notes.append(f"Using image: {self.image_ref}")
            self.setup_notes.append("Container entrypoint metadata: python deepinfer/fit.py")
            self.setup_notes.append(f"Resolved fit script path inside the container: {self.fit_script_path}")
            self.ready = True
        except Exception as exc:
            self.setup_error = (
                "DeepInfer setup failed while preparing the udocker container: "
                f"{type(exc).__name__}: {exc}"
            )
            raise ModelBlockedError(self.setup_error)

    def predict_case(self, case: CaseItem) -> PredictionResult:
        self.ensure_ready()
        if not self.fit_script_path:
            raise RuntimeError("DeepInfer setup completed without resolving an executable fit.py path.")

        case_dir = ensure_clean_dir(resolve_case_output_dir(self.model_name, case.case_id))
        input_nrrd = case_dir / "input.nrrd"
        output_nrrd = case_dir / "output_label.nrrd"
        raw_prediction_path = case_dir / "raw_prediction.nii.gz"
        standardized_path = case_dir / "prediction_binary_aligned.nii.gz"

        convert_image_format(case.mri_path, input_nrrd)

        result = PredictionResult(model_name=self.model_name, case_id=case.case_id)
        start = time.time()
        self._udocker(
            "run",
            "--workdir=/",
            "--entrypoint=python",
            "-v",
            f"{case_dir}:/data",
            self.container_name,
            self.fit_script_path,
            "--ModelName",
            "prostate-segmenter",
            "--Domain",
            "PROMISE12",
            "--InputVolume",
            "/data/input.nrrd",
            "--OutputLabel",
            "/data/output_label.nrrd",
            "--ProcessingType",
            "Accurate",
            "--Inference",
            "Ensemble",
            "--verbose",
        )
        result.runtime_seconds = time.time() - start

        if not output_nrrd.exists() or output_nrrd.stat().st_size == 0:
            raise FileNotFoundError("DeepInfer did not produce the expected non-empty /data/output_label.nrrd file.")

        convert_image_format(output_nrrd, raw_prediction_path)
        standardization = standardize_prediction_to_gt_space(
            raw_prediction_path,
            case.gt_path,
            standardized_path,
            positive_labels=None,
        )

        result.raw_prediction_path = raw_prediction_path
        result.standardized_prediction_path = standardization["standardized_path"]
        result.warnings = flatten_warning_list(self.setup_notes, standardization["warnings"])
        return result


## Model 3: BAMF AIMI Adapter

Colab route:

- download the official released nnU-Net weights
- discover the task directory programmatically
- run `nnUNet_predict`
- keep the model output in original image space
- standardize and validate the final scored mask in GT space

This is the most practical notebook-native route because the repository's container ultimately runs nnU-Net inference internally.


In [13]:
class BamfAimiAdapter(BaseModelAdapter):
    model_name = "BAMF AIMI"
    weights_url = "https://zenodo.org/record/8290093/files/Task788_Prostate.zip"

    def __init__(self, work_dir: Path):
        super().__init__(work_dir)
        self.nnunet_root = self.work_dir / "nnunet_data"
        self.results_folder = self.nnunet_root / "nnUNet_trained_models"
        self.task_name: Optional[str] = None

    def setup(self) -> None:
        try:
            self.nnunet_root.mkdir(parents=True, exist_ok=True)
            self.results_folder.mkdir(parents=True, exist_ok=True)
            weights_zip = download_file(self.weights_url, self.work_dir / "Task788_Prostate.zip", force=False)
            extract_archive(weights_zip, self.results_folder / "nnUNet", force=False)

            candidate_tasks = sorted(
                {path.name for path in (self.results_folder / "nnUNet").rglob("Task788*") if path.is_dir()}
            )
            if not candidate_tasks:
                raise RuntimeError("No Task788 task directory was found after extracting the BAMF weights.")

            preferred = next((name for name in candidate_tasks if name == "Task788_ProstateX"), None)
            self.task_name = preferred or candidate_tasks[0]

            os.environ["nnUNet_raw_data_base"] = str(self.nnunet_root / "raw")
            os.environ["nnUNet_preprocessed"] = str(self.nnunet_root / "preprocessed")
            os.environ["RESULTS_FOLDER"] = str(self.results_folder)

            self.setup_notes.append(f"Resolved nnU-Net task name: {self.task_name}")
            self.ready = True
        except Exception as exc:
            self.setup_error = f"{type(exc).__name__}: {exc}"
            raise

    def predict_case(self, case: CaseItem) -> PredictionResult:
        self.ensure_ready()
        case_dir = ensure_clean_dir(resolve_case_output_dir(self.model_name, case.case_id))
        input_dir = case_dir / "input"
        output_dir = case_dir / "raw_output"
        input_dir.mkdir(parents=True, exist_ok=True)
        output_dir.mkdir(parents=True, exist_ok=True)

        nnunet_input = input_dir / "scan_0000.nii.gz"
        convert_image_format(case.mri_path, nnunet_input)

        result = PredictionResult(model_name=self.model_name, case_id=case.case_id)
        case_warnings: list[str] = []
        start = time.time()
        prediction_log = run_command(
            ["nnUNet_predict", "-i", str(input_dir), "-o", str(output_dir), "-t", str(self.task_name)],
            env={**os.environ},
        )
        result.runtime_seconds = time.time() - start

        if "Cannot run postprocessing because the postprocessing file is missing" in prediction_log:
            case_warnings.append("nnunet_postprocessing_metadata_missing_raw_prediction_used")

        raw_prediction_path = output_dir / "scan.nii.gz"
        if not raw_prediction_path.exists():
            raw_prediction_path = locate_prediction_file(output_dir)
        if not raw_prediction_path.exists():
            raise FileNotFoundError("BAMF AIMI did not produce a usable prediction file.")

        standardized_path = case_dir / "prediction_binary_aligned.nii.gz"
        standardization = standardize_prediction_to_gt_space(
            raw_prediction_path,
            case.gt_path,
            standardized_path,
            positive_labels=None,
        )

        result.raw_prediction_path = raw_prediction_path
        result.standardized_prediction_path = standardization["standardized_path"]
        result.warnings = flatten_warning_list(self.setup_notes, case_warnings, standardization["warnings"])
        return result


## Model 4: MONAI `prostate_mri_anatomy` Adapter

Colab route:

- download the official MONAI bundle from Hugging Face
- run the bundle inference workflow
- merge all non-background labels into a binary whole-gland mask
- revalidate the final mask in GT space before scoring

This model is zonal by design, so the whole-gland conversion step is explicit and mandatory.


In [14]:
def write_monai_test_csv(dataset_dir: Path, image_name: str = "image.nii.gz") -> Path:
    csv_path = dataset_dir / "test.csv"
    csv_path.write_text(f"t2\n{image_name}\n", encoding="utf-8")
    return csv_path


def locate_monai_bundle_prediction(output_dir: Path) -> Path:
    candidates = sorted(
        p
        for p in output_dir.rglob("*")
        if p.is_file() and any(p.name.lower().endswith(ext) for ext in (".nii.gz", ".nii"))
    )
    if not candidates:
        raise FileNotFoundError(f"No MONAI prediction file was found under {output_dir}")

    preferred = [
        path
        for path in candidates
        if any(token in path.name.lower() for token in ("pred", "seg", "label", "trans"))
    ]
    return preferred[0] if preferred else candidates[0]


class MonaiProstateAnatomyAdapter(BaseModelAdapter):
    model_name = "MONAI prostate_mri_anatomy"

    def __init__(self, work_dir: Path):
        super().__init__(work_dir)
        self.bundle_dir = self.work_dir / "prostate_mri_anatomy_bundle"
        self.config_file: Optional[Path] = None
        self.meta_file: Optional[Path] = None
        self.runtime_config_file: Optional[Path] = None

    def setup(self) -> None:
        try:
            snapshot_download(
                repo_id="MONAI/prostate_mri_anatomy",
                revision="0.3.5",
                local_dir=str(self.bundle_dir),
                local_dir_use_symlinks=False,
            )

            self.config_file = self.bundle_dir / "configs" / "inference.json"
            self.meta_file = self.bundle_dir / "configs" / "metadata.json"

            if not self.config_file.exists():
                raise FileNotFoundError(f"Missing MONAI inference config: {self.config_file}")
            if not self.meta_file.exists():
                raise FileNotFoundError(f"Missing MONAI metadata file: {self.meta_file}")
            if not (self.bundle_dir / "models" / "model.pt").exists():
                raise FileNotFoundError(f"Missing MONAI model weights: {self.bundle_dir / 'models' / 'model.pt'}")

            import torch

            cfg = json.loads(self.config_file.read_text(encoding="utf-8"))
            cfg["bundle_root"] = str(self.bundle_dir)
            cfg["dataset_dir"] = "./dataset"
            cfg["output_dir"] = "./eval"
            cfg["datalist"] = (
                "$list(os.path.join(@dataset_dir, rel_path) "
                "for rel_path in pd.read_csv(os.path.join(@dataset_dir, 'test.csv')).t2)"
            )
            cfg["dataloader"]["num_workers"] = 0
            cfg["evaluator"]["amp"] = bool(torch.cuda.is_available())

            self.runtime_config_file = self.bundle_dir / "configs" / "inference_colab_runtime.json"
            self.runtime_config_file.write_text(json.dumps(cfg, indent=2), encoding="utf-8")

            self.setup_notes.append("Bundle input contract preserved: dataset_dir/test.csv with a single 't2' column.")
            self.setup_notes.append("Bundle num_workers reduced to 0 for Colab stability.")
            self.setup_notes.append("Whole-gland benchmark mask is created by merging all non-background bundle labels.")
            self.ready = True
        except Exception as exc:
            self.setup_error = f"{type(exc).__name__}: {exc}"
            raise

    def predict_case(self, case: CaseItem) -> PredictionResult:
        self.ensure_ready()

        case_dir = ensure_clean_dir(resolve_case_output_dir(self.model_name, case.case_id))
        dataset_dir = case_dir / "dataset"
        output_dir = case_dir / "bundle_output"
        dataset_dir.mkdir(parents=True, exist_ok=True)
        output_dir.mkdir(parents=True, exist_ok=True)

        input_file = dataset_dir / "image.nii.gz"
        convert_image_format(case.mri_path, input_file)
        write_monai_test_csv(dataset_dir, input_file.name)

        result = PredictionResult(model_name=self.model_name, case_id=case.case_id)
        start = time.time()

        run_command(
            [
                sys.executable,
                "-m",
                "monai.bundle",
                "run",
                "--run_id",
                "run",
                "--meta_file",
                str(self.meta_file),
                "--config_file",
                str(self.runtime_config_file),
                "--bundle_root",
                str(self.bundle_dir),
                "--dataset_dir",
                str(dataset_dir),
                "--output_dir",
                str(output_dir),
            ]
        )

        result.runtime_seconds = time.time() - start

        raw_bundle_prediction = locate_monai_bundle_prediction(output_dir)
        raw_binary_prediction = case_dir / "raw_whole_gland_binary.nii.gz"

        raw_image = load_image(raw_bundle_prediction)
        merged_binary = binary_image_from_labels(raw_image, positive_labels=None)
        save_image(merged_binary, raw_binary_prediction)

        standardized_path = case_dir / "prediction_binary_aligned.nii.gz"
        standardization = standardize_prediction_to_gt_space(
            raw_binary_prediction,
            case.gt_path,
            standardized_path,
            positive_labels={1},
        )

        result.raw_prediction_path = raw_binary_prediction
        result.standardized_prediction_path = standardization["standardized_path"]
        result.warnings = flatten_warning_list(self.setup_notes, standardization["warnings"])
        return result


## Unified Evaluation Loop

This section runs the four adapters over every paired case and records results in a structured table.

Design choices:

- setup is separated from inference
- any setup failure is preserved explicitly
- each successful prediction is standardized before scoring
- per-case rows include warnings and optional failure reasons
- predictions are saved to disk for later inspection and report generation


In [15]:
import importlib
import importlib.util
from pathlib import Path


model_restore_path = Path("/content/nnUNet_v1/nnunet/training/model_restore.py")
if not model_restore_path.exists():
    raise FileNotFoundError(f"nnU-Net v1 file not found: {model_restore_path}")

text = model_restore_path.read_text(encoding="utf-8")
old = "torch.load(i, map_location=torch.device('cpu'))"
new = "torch.load(i, map_location=torch.device('cpu'), weights_only=False)"

if old in text:
    model_restore_path.write_text(text.replace(old, new), encoding="utf-8")
    print("Patched nnU-Net v1 model_restore.py for PyTorch >= 2.6 checkpoint loading.")
else:
    print("nnU-Net v1 model_restore.py already contains the PyTorch >= 2.6 compatibility patch.")


def module_status_row(name: str, import_name: Optional[str] = None) -> dict[str, Any]:
    import_name = import_name or name
    spec = importlib.util.find_spec(import_name)
    if spec is None:
        return {"package": name, "present": False, "version": "<missing>", "path": "<missing>"}
    module = importlib.import_module(import_name)
    location = getattr(module, "__file__", "<built-in>")
    return {
        "package": name,
        "present": True,
        "version": getattr(module, "__version__", "unknown"),
        "path": str(Path(location).resolve()) if location != "<built-in>" else location,
    }


def validate_prediction_result(prediction: PredictionResult, case: CaseItem) -> dict[str, Any]:
    if prediction.raw_prediction_path is None:
        raise ValueError(f"{prediction.model_name} returned no raw_prediction_path for case {case.case_id}.")
    if prediction.standardized_prediction_path is None:
        raise ValueError(f"{prediction.model_name} returned no standardized_prediction_path for case {case.case_id}.")

    raw_path = Path(prediction.raw_prediction_path)
    standardized_path = Path(prediction.standardized_prediction_path)
    if not raw_path.exists():
        raise FileNotFoundError(f"{prediction.model_name} raw prediction is missing: {raw_path}")
    if not standardized_path.exists():
        raise FileNotFoundError(f"{prediction.model_name} standardized prediction is missing: {standardized_path}")

    standardized_image = load_image(standardized_path)
    gt_image = load_image(case.gt_path)
    if not same_geometry(standardized_image, gt_image):
        raise ValueError(
            f"{prediction.model_name} standardized prediction geometry does not match the GT for case {case.case_id}."
        )

    array = sitk.GetArrayFromImage(standardized_image)
    unique_values = sorted(int(v) for v in np.unique(array))
    if set(unique_values) - {0, 1}:
        raise ValueError(
            f"{prediction.model_name} standardized prediction is not binary for case {case.case_id}. "
            f"Unique values: {unique_values}"
        )

    return {
        "binary_values": unique_values,
        "positive_voxels": int(np.count_nonzero(array)),
        "raw_prediction_path": raw_path,
        "standardized_prediction_path": standardized_path,
    }


Patched nnU-Net v1 model_restore.py for PyTorch >= 2.6 checkpoint loading.


In [21]:
adapters = [
    DzaridisAdapter(MODELS_DIR / "dzaridis"),
    DeepInferAdapter(MODELS_DIR / "deepinfer"),
    BamfAimiAdapter(MODELS_DIR / "bamf_aimi"),
    MonaiProstateAnatomyAdapter(MODELS_DIR / "monai_prostate_mri_anatomy"),
]

setup_rows = []
for adapter in adapters:
    start = time.time()
    try:
        adapter.setup()
        setup_rows.append(
            {
                "model_name": adapter.model_name,
                "status": "ready",
                "setup_seconds": round(time.time() - start, 2),
                "notes": " | ".join(adapter.setup_notes) if adapter.setup_notes else "",
            }
        )
    except Exception as exc:
        adapter.ready = False
        adapter.setup_error = adapter.setup_error or f"{type(exc).__name__}: {exc}"
        setup_rows.append(
            {
                "model_name": adapter.model_name,
                "status": "setup_failed",
                "setup_seconds": round(time.time() - start, 2),
                "notes": adapter.setup_error,
            }
        )

setup_df = pd.DataFrame(setup_rows)
display(setup_df)


$ git lfs install
Updated git hooks.
Git LFS initialized.
$ git lfs pull
$ udocker --allow-root install
$ udocker --allow-root pull deepinfer/prostate:latest
Info: downloading layer sha256:45b9bfad205b6e46ef2fe081f1f6d4961cf9076d0857024f244b0e34e4f08b4d
Info: downloading layer sha256:bfdde49c7035335e6ab62c8a5a109627af7158f29db9fd59dc594b93de9b74bf
Info: downloading layer sha256:6b8a17f3016b88e1508d662a7e8c353de18157d5fa85d15d2f692860a5d9ed0b
Info: downloading layer sha256:116af5ffb2bc1dab44eea6d7166bc9ba69a11ad3589ef6f29af8cc6a54a99362
Info: downloading layer sha256:1ab1f66592ff1476e5f2870efc70ba88ec3ff0eb93926de12f4fa7542c990c8c
Info: downloading layer sha256:e37f749573972a39f8caee79f5b4e8fad7adcaebcf96cac1ff88daeed2d6c7da
Info: downloading layer sha256:abfbe04b33bf08ed83d57db830a36be71d76b1c156da8bed0819013a2bfc898c
Info: downloading layer sha256:0f6b21f7e5f8423ec6cc6ca3c327c5e535d85e6df7bd78fc3df1cf03ff1a35ec
Info: downloading layer sha256:3ed1b364a896020b60ac73071675ab411f40f04e9c7

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

,model_name,status,setup_seconds,notes
0,Dzaridis,ready,0.05,NIfTI-only Colab path enabled; optional DICOM/...
1,DeepInfer,ready,13.89,Using image: deepinfer/prostate:latest | Conta...
2,BAMF AIMI,ready,0.00,Resolved nnU-Net task name: Task788_ProstateX
3,MONAI prostate_mri_anatomy,ready,0.15,Bundle input contract preserved: dataset_dir/t...


In [22]:
package_df = pd.DataFrame(
    [
        module_status_row("numpy"),
        module_status_row("scipy"),
        module_status_row("pandas"),
        module_status_row("SimpleITK"),
        module_status_row("nibabel"),
        module_status_row("nrrd"),
        module_status_row("pydicom"),
        module_status_row("monai"),
        module_status_row("ignite"),
        module_status_row("fire"),
        module_status_row("MedProIO"),
        module_status_row("nnunetv2"),
    ]
)
display(package_df)

artifact_rows = []
for adapter in adapters:
    if isinstance(adapter, DzaridisAdapter):
        artifact_rows.append(
            {
                "model_name": adapter.model_name,
                "ready": adapter.ready,
                "details": (
                    f"repo_exists={adapter.repo_dir.exists()} | "
                    f"checkpoints={len(recursive_find_first_checkpoint(adapter.repo_dir)) if adapter.repo_dir.exists() else 0}"
                ),
            }
        )
    elif isinstance(adapter, DeepInferAdapter):
        artifact_rows.append(
            {
                "model_name": adapter.model_name,
                "ready": adapter.ready,
                "details": (
                    f"udocker={bool(shutil.which('udocker'))} | "
                    f"image_ref={adapter.image_ref or '<unset>'} | "
                    f"fit_script={adapter.fit_script_path or '<unset>'} | "
                    f"inspect_metadata={bool(adapter.container_inspect_output.strip())}"
                ),
            }
        )
    elif isinstance(adapter, BamfAimiAdapter):
        bamf_root = adapter.results_folder / "nnUNet"
        artifact_rows.append(
            {
                "model_name": adapter.model_name,
                "ready": adapter.ready,
                "details": (
                    f"weights_root_exists={bamf_root.exists()} | "
                    f"task_name={adapter.task_name or '<unset>'}"
                ),
            }
        )
    elif isinstance(adapter, MonaiProstateAnatomyAdapter):
        artifact_rows.append(
            {
                "model_name": adapter.model_name,
                "ready": adapter.ready,
                "details": (
                    f"bundle_exists={adapter.bundle_dir.exists()} | "
                    f"inference_json={bool(adapter.config_file and adapter.config_file.exists())} | "
                    f"model_pt={bool((adapter.bundle_dir / 'models' / 'model.pt').exists())} | "
                    f"runtime_config={bool(adapter.runtime_config_file and adapter.runtime_config_file.exists())}"
                ),
            }
        )

diagnostics_df = pd.DataFrame(artifact_rows)
display(diagnostics_df)

deepinfer_adapter = next((adapter for adapter in adapters if isinstance(adapter, DeepInferAdapter)), None)
if deepinfer_adapter is not None and deepinfer_adapter.container_inspect_output:
    print("DeepInfer inspect excerpt:")
    print(deepinfer_adapter.container_inspect_output[:1200])


,package,present,version,path
0,numpy,True,1.26.2,/usr/local/lib/python3.12/dist-packages/numpy/...
1,scipy,True,1.13.1,/usr/local/lib/python3.12/dist-packages/scipy/...
2,pandas,True,2.2.2,/usr/local/lib/python3.12/dist-packages/pandas...
3,SimpleITK,True,2.3.1,/usr/local/lib/python3.12/dist-packages/Simple...
4,nibabel,True,5.2.1,/usr/local/lib/python3.12/dist-packages/nibabe...
5,nrrd,True,1.0.0,/usr/local/lib/python3.12/dist-packages/nrrd/_...
6,pydicom,True,2.4.4,/usr/local/lib/python3.12/dist-packages/pydico...
7,monai,True,1.4.0,/usr/local/lib/python3.12/dist-packages/monai/...
8,ignite,True,0.4.11,/usr/local/lib/python3.12/dist-packages/ignite...
9,fire,True,0.6.0,/usr/local/lib/python3.12/dist-packages/fire/_...


,model_name,ready,details
0,Dzaridis,True,repo_exists=True | checkpoints=4
1,DeepInfer,True,udocker=True | image_ref=deepinfer/prostate:la...
2,BAMF AIMI,True,weights_root_exists=True | task_name=Task788_P...
3,MONAI prostate_mri_anatomy,True,bundle_exists=True | inference_json=True | mod...


DeepInfer inspect excerpt:
{
    "architecture": "amd64",
    "author": "Alireza Mehrtash <mehrtash@bwh.harvard.edu>",
    "config": {
        "ArgsEscaped": true,
        "AttachStderr": false,
        "AttachStdin": false,
        "AttachStdout": false,
        "Cmd": null,
        "Domainname": "",
        "Entrypoint": [
            "python",
            "deepinfer/fit.py"
        ],
        "Env": [
            "PATH=/opt/conda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin",
            "LANG=C.UTF-8",
            "LC_ALL=C.UTF-8",
            "CONDA_DIR=/opt/conda"
        ],
        "Hostname": "73229862783c",
        "Image": "sha256:cf0c86d21c4aeba057bb541ec085c921aa595b17bfae1f54a1e8d892004ecb35",
        "Labels": {},
        "OnBuild": [],
        "OpenStdin": false,
        "StdinOnce": false,
        "Tty": false,
        "User": "",
        "Volumes": null,
        "WorkingDir": ""
    },
    "container": "ddf9102a9377306a52c00514596cdbffa26c1a5cf561b7

In [23]:
if not cases:
    raise RuntimeError("No paired PROMISE12 cases were discovered, so the smoke test cannot run.")

pilot_case = cases[0]
smoke_rows = []

for adapter in adapters:
    row = {
        "case_id": pilot_case.case_id,
        "model_name": adapter.model_name,
        "status": "failed",
        "runtime_seconds": np.nan,
        "warnings": "",
        "failure_reason": "",
        "binary_values": "",
        "positive_voxels": np.nan,
        "raw_prediction_path": "",
        "standardized_prediction_path": "",
    }

    if not adapter.ready:
        row["status"] = "setup_failed"
        row["failure_reason"] = adapter.setup_error or "Model setup did not complete."
        smoke_rows.append(row)
        continue

    try:
        prediction = adapter.predict_case(pilot_case)
        validation = validate_prediction_result(prediction, pilot_case)
        row.update(
            {
                "status": "success",
                "runtime_seconds": prediction.runtime_seconds,
                "warnings": " | ".join(prediction.warnings),
                "binary_values": ",".join(str(v) for v in validation["binary_values"]),
                "positive_voxels": validation["positive_voxels"],
                "raw_prediction_path": str(validation["raw_prediction_path"]),
                "standardized_prediction_path": str(validation["standardized_prediction_path"]),
            }
        )
    except Exception as exc:
        row["status"] = "failed"
        row["failure_reason"] = f"{type(exc).__name__}: {exc}"
        print(f"[SMOKE TEST] {adapter.model_name} failed on {pilot_case.case_id}")
        print(traceback.format_exc())

    smoke_rows.append(row)

smoke_test_df = pd.DataFrame(smoke_rows)
display(smoke_test_df)

SMOKE_TEST_PASSED = bool(smoke_test_df["status"].eq("success").all())
if not SMOKE_TEST_PASSED:
    failing_models = ", ".join(smoke_test_df.loc[smoke_test_df["status"] != "success", "model_name"].tolist())
    raise RuntimeError(
        "Smoke test failed. The full benchmark is blocked until every adapter succeeds on the pilot case. "
        f"Failing models: {failing_models}"
    )

print(f"Smoke test passed on pilot case: {pilot_case.case_id}")


There are 1 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 1 cases that I would like to predict

Predicting ProstateWG_Case00:
perform_everything_on_gpu: True
Prediction done, transferring to CPU if needed
sending off prediction to background worker for resampling and export
done with ProstateWG_Case00
$ udocker --allow-root run --workdir=/ --entrypoint=python -v /content/promise12_benchmark/outputs/predictions/deepinfer/case00:/data deepinfer_prostate /deepinfer/fit.py --ModelName prostate-segmenter --Domain PROMISE12 --InputVolume /data/input.nrrd --OutputLabel /data/output_label.nrrd --ProcessingType Accurate --Inference Ensemble --verbose
2026-04-03 18:15:30.320365: W tensorflow/core/platform/cpu_feature_guard.cc:45] The TensorFlow library wasn't compiled to use SSE4.1 instructions, but these are available on your machine and could speed up CPU computations.
2026-04-03 18:15:30.320611: W tensorflow/core/platform/cpu_fea

,case_id,model_name,status,runtime_seconds,warnings,failure_reason,binary_values,positive_voxels,raw_prediction_path,standardized_prediction_path
0,Case00,Dzaridis,success,11.177379,Discovered 4 checkpoint file(s). | NIfTI-only ...,,"0,1",66532,/content/promise12_benchmark/models/dzaridis/M...,/content/promise12_benchmark/outputs/predictio...
1,Case00,DeepInfer,success,53.712625,Container entrypoint metadata: python deepinfe...,,"0,1",68827,/content/promise12_benchmark/outputs/predictio...,/content/promise12_benchmark/outputs/predictio...
2,Case00,BAMF AIMI,success,19.976596,Resolved nnU-Net task name: Task788_ProstateX ...,,"0,1",69431,/content/promise12_benchmark/outputs/predictio...,/content/promise12_benchmark/outputs/predictio...
3,Case00,MONAI prostate_mri_anatomy,success,20.349495,Bundle input contract preserved: dataset_dir/t...,,"0,1",73944,/content/promise12_benchmark/outputs/predictio...,/content/promise12_benchmark/outputs/predictio...


Smoke test passed on pilot case: Case00


In [24]:
if not globals().get("SMOKE_TEST_PASSED", False):
    raise RuntimeError("Run the smoke-test cell successfully before launching the full evaluation loop.")

all_results: list[dict[str, Any]] = []
prediction_registry: dict[tuple[str, str], Path] = {}

for case in tqdm(cases, desc="Evaluating cases"):
    for adapter in adapters:
        base_row = {
            "case_id": case.case_id,
            "model_name": adapter.model_name,
            "status": "failed",
            "dice": np.nan,
            "tversky": np.nan,
            "non_intersect_pct": np.nan,
            "runtime_seconds": np.nan,
            "warnings": "",
            "failure_reason": "",
            "raw_prediction_path": "",
            "standardized_prediction_path": "",
        }

        if not adapter.ready:
            base_row["status"] = "setup_failed"
            base_row["failure_reason"] = adapter.setup_error or "Model setup did not complete."
            all_results.append(base_row)
            continue

        try:
            prediction = adapter.predict_case(case)
            validation = validate_prediction_result(prediction, case)
            metrics = compute_binary_metrics(validation["standardized_prediction_path"], case.gt_path)
            base_row.update(
                {
                    "status": "success",
                    "dice": metrics["dice"],
                    "tversky": metrics["tversky"],
                    "non_intersect_pct": metrics["non_intersect_pct"],
                    "runtime_seconds": prediction.runtime_seconds,
                    "warnings": " | ".join(flatten_warning_list(prediction.warnings, metrics["warnings"])),
                    "raw_prediction_path": str(validation["raw_prediction_path"]),
                    "standardized_prediction_path": str(validation["standardized_prediction_path"]),
                }
            )
            prediction_registry[(case.case_id, adapter.model_name)] = validation["standardized_prediction_path"]
        except Exception as exc:
            base_row["status"] = "failed"
            base_row["failure_reason"] = f"{type(exc).__name__}: {exc}"
            base_row["warnings"] = "prediction_failed"
            print(f"[FULL EVAL] {adapter.model_name} failed on {case.case_id}")
            print(traceback.format_exc())

        all_results.append(base_row)

results_df = pd.DataFrame(all_results)
per_case_csv_path = TABLES_DIR / "per_case_results.csv"
results_df.to_csv(per_case_csv_path, index=False)

print(f"Saved per-case results to: {per_case_csv_path}")
display(results_df.head(20))


Evaluating cases:   0%|          | 0/30 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
2026-04-03 18:24:07,508 - INFO - > config_file: '/content/promise12_benchmark/models/monai_prostate_mri_anatomy/prostate_mri_anatomy_bundle/configs/inference_colab_runtime.json'
2026-04-03 18:24:07,508 - INFO - > meta_file: '/content/promise12_benchmark/models/monai_prostate_mri_anatomy/prostate_mri_anatomy_bundle/configs/metadata.json'
2026-04-03 18:24:07,508 - INFO - > run_id: 'run'
2026-04-03 18:24:07,509 - INFO - > bundle_root: '/content/promise12_benchmark/models/monai_prostate_mri_anatomy/prostate_mri_anatomy_bundle'
2026-04-03 18:24:07,509 - INFO - > dataset_dir: '/content/promise12_benchmark/outputs/predictions/monai_prostate_mri_anatomy/case03/dataset'
2026-04-03 18:24:07,509 - INFO - > output_dir: '/content/promise12_benchmark/outputs/predictions/monai_prostate_mri_anatomy/case03/bundle_output'
2026-04-03 18:24:07,509 - INFO - ---


2026-04-03 18:24:07,509 - INFO - Setting logging properties based on config: /content/promise1

,case_id,model_name,status,dice,tversky,non_intersect_pct,runtime_seconds,warnings,failure_reason,raw_prediction_path,standardized_prediction_path
0,Case00,Dzaridis,success,0.955070,0.947407,2.522095,10.035993,Discovered 4 checkpoint file(s). | NIfTI-only ...,,/content/promise12_benchmark/models/dzaridis/M...,/content/promise12_benchmark/outputs/predictio...
1,Case00,DeepInfer,success,0.933753,0.932535,6.318741,53.363000,Container entrypoint metadata: python deepinfe...,,/content/promise12_benchmark/outputs/predictio...,/content/promise12_benchmark/outputs/predictio...
2,Case00,BAMF AIMI,success,0.962086,0.962511,3.897395,20.024635,Resolved nnU-Net task name: Task788_ProstateX ...,,/content/promise12_benchmark/outputs/predictio...,/content/promise12_benchmark/outputs/predictio...
3,Case00,MONAI prostate_mri_anatomy,success,0.937719,0.950100,9.186682,20.268481,Bundle input contract preserved: dataset_dir/t...,,/content/promise12_benchmark/outputs/predictio...,/content/promise12_benchmark/outputs/predictio...
4,Case01,Dzaridis,success,0.928749,0.917054,4.066774,10.107474,Discovered 4 checkpoint file(s). | NIfTI-only ...,,/content/promise12_benchmark/models/dzaridis/M...,/content/promise12_benchmark/outputs/predictio...
5,Case01,DeepInfer,success,0.889031,0.871162,6.291553,54.289645,Container entrypoint metadata: python deepinfe...,,/content/promise12_benchmark/outputs/predictio...,/content/promise12_benchmark/outputs/predictio...
6,Case01,BAMF AIMI,success,0.912189,0.909662,8.143271,20.145516,Resolved nnU-Net task name: Task788_ProstateX ...,,/content/promise12_benchmark/outputs/predictio...,/content/promise12_benchmark/outputs/predictio...
7,Case01,MONAI prostate_mri_anatomy,success,0.911808,0.922730,11.439693,20.183626,Bundle input contract preserved: dataset_dir/t...,,/content/promise12_benchmark/outputs/predictio...,/content/promise12_benchmark/outputs/predictio...
8,Case02,Dzaridis,success,0.940477,0.923341,1.376663,10.059414,Discovered 4 checkpoint file(s). | NIfTI-only ...,,/content/promise12_benchmark/models/dzaridis/M...,/content/promise12_benchmark/outputs/predictio...
9,Case02,DeepInfer,success,0.915752,0.903782,5.288920,54.021795,Container entrypoint metadata: python deepinfe...,,/content/promise12_benchmark/outputs/predictio...,/content/promise12_benchmark/outputs/predictio...


## Per-Case Results Table

The `results_df` table is the detailed case-level record. Each row corresponds to one `(case, model)` pair and includes:

- case identifier
- model name
- the three requested metrics
- runtime in seconds when available
- warnings such as resampling or empty predictions
- a failure reason if setup or inference did not complete


In [25]:
display(
    results_df.sort_values(["case_id", "model_name"])
    .reset_index(drop=True)
    .style.format(
        {
            "dice": "{:.4f}",
            "tversky": "{:.4f}",
            "non_intersect_pct": "{:.2f}",
            "runtime_seconds": "{:.2f}",
        },
        na_rep="NA",
    )
)


,case_id,model_name,status,dice,tversky,non_intersect_pct,runtime_seconds,warnings,failure_reason,raw_prediction_path,standardized_prediction_path
0,Case00,BAMF AIMI,success,0.9621,0.9625,3.90,20.02,Resolved nnU-Net task name: Task788_ProstateX | nnunet_postprocessing_metadata_missing_raw_prediction_used,,/content/promise12_benchmark/outputs/predictions/bamf_aimi/case00/raw_output/scan.nii.gz,/content/promise12_benchmark/outputs/predictions/bamf_aimi/case00/prediction_binary_aligned.nii.gz
1,Case00,DeepInfer,success,0.9338,0.9325,6.32,53.36,Container entrypoint metadata: python deepinfer/fit.py | Resolved fit script path inside the container: /deepinfer/fit.py | Using image: deepinfer/prostate:latest,,/content/promise12_benchmark/outputs/predictions/deepinfer/case00/raw_prediction.nii.gz,/content/promise12_benchmark/outputs/predictions/deepinfer/case00/prediction_binary_aligned.nii.gz
2,Case00,Dzaridis,success,0.9551,0.9474,2.52,10.04,Discovered 4 checkpoint file(s). | NIfTI-only Colab path enabled; optional DICOM/Orthanc export is bypassed. | Whole-gland benchmark output is generated directly from the repo's WG stage.,,/content/promise12_benchmark/models/dzaridis/MRI-Prostate-Gland-and-Zone-Segmentor/Outputs/Case00/Resampled/wg_binary.nii.gz,/content/promise12_benchmark/outputs/predictions/dzaridis/case00/prediction_binary_aligned.nii.gz
3,Case00,MONAI prostate_mri_anatomy,success,0.9377,0.9501,9.19,20.27,Bundle input contract preserved: dataset_dir/test.csv with a single 't2' column. | Bundle num_workers reduced to 0 for Colab stability. | Whole-gland benchmark mask is created by merging all non-background bundle labels.,,/content/promise12_benchmark/outputs/predictions/monai_prostate_mri_anatomy/case00/raw_whole_gland_binary.nii.gz,/content/promise12_benchmark/outputs/predictions/monai_prostate_mri_anatomy/case00/prediction_binary_aligned.nii.gz
4,Case01,BAMF AIMI,success,0.9122,0.9097,8.14,20.15,Resolved nnU-Net task name: Task788_ProstateX | nnunet_postprocessing_metadata_missing_raw_prediction_used,,/content/promise12_benchmark/outputs/predictions/bamf_aimi/case01/raw_output/scan.nii.gz,/content/promise12_benchmark/outputs/predictions/bamf_aimi/case01/prediction_binary_aligned.nii.gz
5,Case01,DeepInfer,success,0.8890,0.8712,6.29,54.29,Container entrypoint metadata: python deepinfer/fit.py | Resolved fit script path inside the container: /deepinfer/fit.py | Using image: deepinfer/prostate:latest,,/content/promise12_benchmark/outputs/predictions/deepinfer/case01/raw_prediction.nii.gz,/content/promise12_benchmark/outputs/predictions/deepinfer/case01/prediction_binary_aligned.nii.gz
6,Case01,Dzaridis,success,0.9287,0.9171,4.07,10.11,Discovered 4 checkpoint file(s). | NIfTI-only Colab path enabled; optional DICOM/Orthanc export is bypassed. | Whole-gland benchmark output is generated directly from the repo's WG stage.,,/content/promise12_benchmark/models/dzaridis/MRI-Prostate-Gland-and-Zone-Segmentor/Outputs/Case01/Resampled/wg_binary.nii.gz,/content/promise12_benchmark/outputs/predictions/dzaridis/case01/prediction_binary_aligned.nii.gz
7,Case01,MONAI prostate_mri_anatomy,success,0.9118,0.9227,11.44,20.18,Bundle input contract preserved: dataset_dir/test.csv with a single 't2' column. | Bundle num_workers reduced to 0 for Colab stability. | Whole-gland benchmark mask is created by merging all non-background bundle labels.,,/content/promise12_benchmark/outputs/predictions/monai_prostate_mri_anatomy/case01/raw_whole_gland_binary.nii.gz,/content/promise12_benchmark/outputs/predictions/monai_prostate_mri_anatomy/case01/prediction_binary_aligned.nii.gz
8,Case02,BAMF AIMI,success,0.9445,0.9380,3.87,20.09,Resolved nnU-Net task name: Task788_ProstateX | nnunet_postprocessing_metadata_missing_raw_prediction_used,,/content/promise12_benchmark/outputs/predictions/bamf_aimi/case02/raw_output/scan.nii.gz,/content/promise12_benchmark/outputs/predictions/bamf_aimi/case02/prediction_binary_aligned.nii.gz
9,Case02,DeepInfer,success,0.9158,0.

## Final Summary Table

The summary table compresses the benchmark to one row per model.

Required averaging policy:

- `Dice_all`: average Dice across all successfully scored cases, including zeros
- `Dice_nonzero`: average Dice after excluding zero-score cases only
- `Tversky_all`: average Tversky across all successfully scored cases, including zeros
- `Tversky_nonzero`: average Tversky after excluding zero-score cases only
- `NonIntersectPct_all`: average non-intersecting prediction percentage across all successfully scored cases, keeping zeros

Setup or inference failures remain explicit failure counts instead of being silently converted into metric zeros.


In [26]:
successful_results = results_df[results_df["status"] == "success"].copy()


def build_zero_case_notes(group: pd.DataFrame) -> str:
    notes: list[str] = []
    for _, row in group.sort_values("case_id").iterrows():
        pieces = []
        if pd.notna(row["dice"]) and math.isclose(float(row["dice"]), 0.0, abs_tol=1e-12):
            pieces.append("Dice=0")
        if pd.notna(row["tversky"]) and math.isclose(float(row["tversky"]), 0.0, abs_tol=1e-12):
            pieces.append("Tversky=0")
        if pieces:
            notes.append(f"{row['case_id']}: {', '.join(pieces)}")
    return " | ".join(notes)


summary_rows = []
for adapter in adapters:
    model_name = adapter.model_name
    model_success = successful_results[successful_results["model_name"] == model_name].copy()
    model_all = results_df[results_df["model_name"] == model_name].copy()

    summary_rows.append(
        {
            "model": model_name,
            "n_cases_total": int(len(model_all)),
            "n_scored_cases": int(len(model_success)),
            "n_failed_cases": int((model_all["status"] != "success").sum()),
            "n_zero_dice": int((model_success["dice"] == 0).sum()) if not model_success.empty else 0,
            "n_zero_tversky": int((model_success["tversky"] == 0).sum()) if not model_success.empty else 0,
            "Dice_all": float(model_success["dice"].mean()) if not model_success.empty else np.nan,
            "Dice_nonzero": float(model_success.loc[model_success["dice"] > 0, "dice"].mean()) if (model_success["dice"] > 0).any() else np.nan,
            "Tversky_all": float(model_success["tversky"].mean()) if not model_success.empty else np.nan,
            "Tversky_nonzero": float(model_success.loc[model_success["tversky"] > 0, "tversky"].mean()) if (model_success["tversky"] > 0).any() else np.nan,
            "NonIntersectPct_all": float(model_success["non_intersect_pct"].mean()) if not model_success.empty else np.nan,
            "zero_case_notes": build_zero_case_notes(model_success),
            "failure_notes": " | ".join(
                sorted(
                    dict.fromkeys(
                        reason
                        for reason in model_all["failure_reason"].astype(str).tolist()
                        if reason and reason != "nan"
                    )
                )
            ),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_csv_path = TABLES_DIR / "summary_results.csv"
summary_df.to_csv(summary_csv_path, index=False)

print(f"Saved summary results to: {summary_csv_path}")
display(summary_df)


Saved summary results to: /content/promise12_benchmark/outputs/tables/summary_results.csv


,model,n_cases_total,n_scored_cases,n_failed_cases,n_zero_dice,n_zero_tversky,Dice_all,Dice_nonzero,Tversky_all,Tversky_nonzero,NonIntersectPct_all,zero_case_notes,failure_notes
0,Dzaridis,30,29,1,0,0,0.900233,0.900233,0.884400,0.884400,5.450352,,ValueError: Dzaridis standardized prediction g...
1,DeepInfer,30,29,1,0,0,0.837564,0.837564,0.816327,0.816327,7.305648,,ValueError: DeepInfer standardized prediction ...
2,BAMF AIMI,30,29,1,0,0,0.916734,0.916734,0.917247,0.917247,8.190606,,ValueError: BAMF AIMI standardized prediction ...
3,MONAI prostate_mri_anatomy,30,29,1,0,0,0.815410,0.815410,0.832783,0.832783,21.946745,,ValueError: MONAI prostate_mri_anatomy standar...


## Best-Case Selection for Qualitative Visualization

The notebook uses one case for the qualitative figure, but it is not chosen manually or randomly.

Selection rule:

1. For each successful `(case, model)` pair compute  
   `composite = (Dice + Tversky + (1 - NonIntersectPct/100)) / 3`
2. Average that composite score across models for each case
3. Prefer cases with the largest number of successful model outputs
4. Select the highest-ranked case
5. Use the axial slice with the largest GT mask area


In [27]:
if successful_results.empty:
    raise RuntimeError("No successful model predictions are available, so the qualitative comparison figure cannot be created.")

successful_results = successful_results.copy()
successful_results["composite_score"] = (
    successful_results["dice"]
    + successful_results["tversky"]
    + (1.0 - successful_results["non_intersect_pct"] / 100.0)
) / 3.0

case_rank_df = (
    successful_results.groupby("case_id", as_index=False)
    .agg(avg_composite_score=("composite_score", "mean"), n_models_success=("model_name", "nunique"))
    .sort_values(["n_models_success", "avg_composite_score"], ascending=[False, False])
    .reset_index(drop=True)
)

best_case_id = str(case_rank_df.iloc[0]["case_id"])
best_case = next(case for case in cases if case.case_id == best_case_id)
gt_array = sitk.GetArrayFromImage(load_image(best_case.gt_path)).astype(bool)
slice_areas = gt_array.sum(axis=(1, 2))
best_slice_index = int(np.argmax(slice_areas))

print("Chosen qualitative case:", best_case_id)
print("Models available for that case:", int(case_rank_df.iloc[0]["n_models_success"]))
print("Selected axial slice index:", best_slice_index)
display(case_rank_df.head(10))


Chosen qualitative case: Case00
Models available for that case: 4
Selected axial slice index: 10


,case_id,avg_composite_score,n_models_success
0,Case00,0.946828,4
1,Case03,0.946278,4
2,Case02,0.937756,4
3,Case04,0.932100,4
4,Case19,0.922666,4
5,Case05,0.921974,4
6,Case17,0.920581,4
7,Case18,0.920002,4
8,Case01,0.913581,4
9,Case21,0.909480,4


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


## Qualitative Figure Generation

The main figure is intentionally simple and report-friendly:

- rows correspond to models
- columns are `[MRI] [Ground Truth] [Prediction]`
- masks are shown as standalone binary images, not overlays
- all rows use the same case and same slice index
- MRI display scaling is kept consistent across rows


In [28]:
def image_to_axial_slice(image: sitk.Image, slice_index: int) -> np.ndarray:
    array = sitk.GetArrayFromImage(image)
    if slice_index < 0 or slice_index >= array.shape[0]:
        raise IndexError(f"Slice index {slice_index} is out of bounds for volume with {array.shape[0]} slices.")
    return array[slice_index]


def format_metric_triplet(row: pd.Series) -> str:
    if row is None or row.empty or row.get("status") != "success":
        return "Prediction unavailable"
    return f"Dice={row['dice']:.3f}\nTversky={row['tversky']:.3f}\nNonIntersect={row['non_intersect_pct']:.2f}%"


display_mri_image = align_image_for_display(best_case.mri_path, best_case.gt_path)
display_gt_image = load_image(best_case.gt_path)

display_mri_slice = image_to_axial_slice(display_mri_image, best_slice_index)
display_gt_slice = image_to_axial_slice(display_gt_image, best_slice_index)

vmin = float(np.percentile(display_mri_slice, 1))
vmax = float(np.percentile(display_mri_slice, 99))

figure_path = FIGURES_DIR / "best_case_comparison.png"
n_rows = len(adapters)
fig, axes = plt.subplots(n_rows, 3, figsize=(14, 3.6 * n_rows), constrained_layout=True)
if n_rows == 1:
    axes = np.array([axes])

for row_index, adapter in enumerate(adapters):
    row_axes = axes[row_index]
    row_axes[0].imshow(display_mri_slice, cmap="gray", vmin=vmin, vmax=vmax)
    row_axes[1].imshow(display_gt_slice, cmap="gray")

    row_axes[0].set_ylabel(adapter.model_name, rotation=90, fontsize=11, labelpad=20)
    if row_index == 0:
        row_axes[0].set_title("Original MRI")
        row_axes[1].set_title("Ground Truth")
        row_axes[2].set_title("Prediction")

    row_axes[0].axis("off")
    row_axes[1].axis("off")
    row_axes[2].axis("off")

    selected_row = results_df[(results_df["case_id"] == best_case.case_id) & (results_df["model_name"] == adapter.model_name)]
    selected_row = selected_row.iloc[0] if not selected_row.empty else pd.Series(dtype=object)

    if selected_row.empty or selected_row.get("status") != "success":
        row_axes[2].text(0.5, 0.5, "Prediction\nunavailable", ha="center", va="center", fontsize=12, transform=row_axes[2].transAxes)
    else:
        pred_image = load_image(Path(selected_row["standardized_prediction_path"]))
        pred_slice = image_to_axial_slice(pred_image, best_slice_index)
        row_axes[2].imshow(pred_slice, cmap="gray")

    row_axes[2].text(
        0.98,
        0.02,
        format_metric_triplet(selected_row if isinstance(selected_row, pd.Series) else pd.Series(dtype=object)),
        ha="right",
        va="bottom",
        fontsize=9,
        color="black",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.9),
        transform=row_axes[2].transAxes,
    )

fig.suptitle(f"PROMISE12 Qualitative Comparison | Case={best_case.case_id} | Slice={best_slice_index}", fontsize=15)
fig.savefig(figure_path, dpi=220, bbox_inches="tight")
plt.show()
print(f"Saved qualitative figure to: {figure_path}")


Saved qualitative figure to: /content/promise12_benchmark/outputs/figures/best_case_comparison.png


## PDF Report Export

The PDF is the main output artifact of this notebook.

It contains:

1. title
2. short experiment setup
3. brief metric descriptions
4. final summary table
5. chosen visualization case information
6. the qualitative comparison figure
7. artifact paths

ReportLab is used so the export works reliably in Colab without LaTeX or browser-print workflows.


In [29]:
def format_summary_for_report(df: pd.DataFrame) -> pd.DataFrame:
    report_df = df.copy()
    numeric_cols = ["Dice_all", "Dice_nonzero", "Tversky_all", "Tversky_nonzero", "NonIntersectPct_all"]
    for col in numeric_cols:
        report_df[col] = report_df[col].map(lambda v: "NA" if pd.isna(v) else f"{v:.4f}")
    integer_cols = ["n_cases_total", "n_scored_cases", "n_failed_cases", "n_zero_dice", "n_zero_tversky"]
    for col in integer_cols:
        report_df[col] = report_df[col].astype(str)
    return report_df


report_pdf_path = REPORTS_DIR / "promise12_four_model_benchmark_report.pdf"
report_summary_df = format_summary_for_report(summary_df)

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="BodySmall", parent=styles["BodyText"], fontSize=9, leading=12, alignment=TA_LEFT))

story = []
story.append(Paragraph("PROMISE12 Whole-Gland Benchmark Report", styles["Title"]))
story.append(Spacer(1, 0.2 * inch))
story.append(
    Paragraph(
        "This report summarizes an end-to-end Colab benchmark of four fixed prostate MRI segmentation models on PROMISE12 test cases with ground-truth whole-gland masks. Each prediction was standardized to a binary whole-gland mask in the ground-truth image space before metrics were computed.",
        styles["BodyText"],
    )
)
story.append(Spacer(1, 0.16 * inch))
story.append(
    Paragraph(
        "<b>Metrics.</b> Dice measures binary overlap. Tversky uses alpha=0.3 and beta=0.7 to penalize false negatives more than false positives. NonIntersectPct measures the percentage of predicted positive voxels that lie outside the ground truth; lower is better.",
        styles["BodyText"],
    )
)
story.append(Spacer(1, 0.18 * inch))
story.append(Paragraph("Summary Comparison Table", styles["Heading2"]))
story.append(Spacer(1, 0.08 * inch))

report_table_columns = ["model", "n_cases_total", "n_scored_cases", "n_failed_cases", "Dice_all", "Dice_nonzero", "Tversky_all", "Tversky_nonzero", "NonIntersectPct_all", "n_zero_dice", "n_zero_tversky"]
report_table_data = [report_table_columns]
for _, row in report_summary_df[report_table_columns].iterrows():
    report_table_data.append([str(row[col]) for col in report_table_columns])

summary_table = Table(report_table_data, repeatRows=1)
summary_table.setStyle(
    TableStyle(
        [
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F4E79")),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("FONTSIZE", (0, 0), (-1, -1), 8),
            ("LEADING", (0, 0), (-1, -1), 10),
            ("GRID", (0, 0), (-1, -1), 0.3, colors.grey),
            ("BACKGROUND", (0, 1), (-1, -1), colors.whitesmoke),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.HexColor("#EAF2F8")]),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ]
    )
)
story.append(summary_table)
story.append(Spacer(1, 0.2 * inch))

story.append(Paragraph("Selected Qualitative Case", styles["Heading2"]))
story.append(
    Paragraph(
        f"Case ID: <b>{best_case.case_id}</b><br/>Selected axial slice: <b>{best_slice_index}</b><br/>Selection rule: highest average composite score across models, with preference for cases having the largest number of successful model outputs.",
        styles["BodyText"],
    )
)
story.append(Spacer(1, 0.12 * inch))
story.append(RLImage(str(figure_path), width=10.2 * inch, height=10.2 * inch * 0.72))
story.append(Spacer(1, 0.16 * inch))
story.append(
    Paragraph(
        f"<b>Artifacts</b><br/>PDF report: {report_pdf_path}<br/>Per-case CSV: {per_case_csv_path}<br/>Summary CSV: {summary_csv_path}<br/>Figure PNG: {figure_path}",
        styles["BodySmall"],
    )
)

doc = SimpleDocTemplate(
    str(report_pdf_path),
    pagesize=landscape(letter),
    leftMargin=0.5 * inch,
    rightMargin=0.5 * inch,
    topMargin=0.45 * inch,
    bottomMargin=0.45 * inch,
)
doc.build(story)
print(f"Saved PDF report to: {report_pdf_path}")


Saved PDF report to: /content/promise12_benchmark/reports/promise12_four_model_benchmark_report.pdf


## Final Artifact Paths

The PDF is the main deliverable. The CSV tables and figure are saved alongside it for debugging and reuse.


In [30]:
artifact_paths = pd.DataFrame(
    [
        {"artifact": "PDF report", "path": str(report_pdf_path)},
        {"artifact": "Per-case CSV", "path": str(per_case_csv_path)},
        {"artifact": "Summary CSV", "path": str(summary_csv_path)},
        {"artifact": "Best-case figure", "path": str(figure_path)},
        {"artifact": "Output root", "path": str(OUTPUTS_DIR)},
    ]
)
display(artifact_paths)


,artifact,path
0,PDF report,/content/promise12_benchmark/reports/promise12...
1,Per-case CSV,/content/promise12_benchmark/outputs/tables/pe...
2,Summary CSV,/content/promise12_benchmark/outputs/tables/su...
3,Best-case figure,/content/promise12_benchmark/outputs/figures/b...
4,Output root,/content/promise12_benchmark/outputs
